# CDV Phylodynamics — Dataset Exploration

Interactive companion to the pipeline scripts. This notebook **imports** the functions
from `scripts/` rather than copying them, so there is one source of truth. Fix a bug in
a script and this notebook picks it up on the next restart.

Use it for the parts that need your eyes: reviewing ambiguous records, deciding scope,
and checking that the dataset can actually support the analysis.

**Order of operations**
1. Run the fetch and curation scripts (cells below, or from the terminal)
2. Explore what came back
3. Work the `needs_review` loop until it's clean
4. Make the scope decision at the Week 3 gate
5. Hand off to alignment and tree building

> The pipeline itself lives in `scripts/`. Don't paste analysis code here permanently —
> if you write something worth keeping, move it into a script so it stays reproducible.

## Setup

**Run this section first.** Every later cell depends on the imports and on `ROOT` being
set here. If you see `NameError: name 'os' is not defined` or similar, the kernel was
restarted and this cell needs re-running — *Kernel → Restart Kernel and Run All Cells*
is the reliable way back.

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Optional override. Leave as None — the cell finds the repo on its own.
REPO_ROOT = None

MARKER = Path("scripts") / "02_curate_metadata.py"

def find_repo_root(explicit=None):
    # 1. explicit override
    if explicit is not None:
        cand = Path(explicit).expanduser().resolve()
        if (cand / MARKER).is_file():
            return cand
        raise FileNotFoundError(f"REPO_ROOT is set to {cand} but {MARKER} isn't there.")

    # 2. cwd and its parents (covers notebook at repo root or in notebooks/)
    cwd = Path.cwd().resolve()
    for cand in [cwd, *cwd.parents]:
        if (cand / MARKER).is_file():
            return cand

    # 3. last resort: search the home directory for it
    home = Path.home()
    hits = [p.parent.parent for p in home.rglob(str(MARKER)) if ".ipynb_checkpoints" not in str(p)]
    hits = sorted(set(hits))
    if len(hits) == 1:
        print(f"note: repo wasn't near the notebook; found it at {hits[0]}")
        return hits[0]
    if len(hits) > 1:
        raise FileNotFoundError(
            "Found more than one copy of the repo. Set REPO_ROOT to the one you want:\n"
            + "\n".join(f'    REPO_ROOT = Path("{h}")' for h in hits)
        )

    # nothing anywhere — say exactly what's wrong and what's on disk
    listing = sorted(p.name for p in cwd.iterdir())[:25] if cwd.is_dir() else []
    raise FileNotFoundError(
        f"Couldn't find scripts/02_curate_metadata.py anywhere under {home}.\n\n"
        f"cwd is {cwd}\n"
        f"cwd contains: {listing}\n\n"
        "The repo folder probably wasn't uploaded, or was uploaded flattened.\n"
        "Expected layout:\n"
        "    <repo>/scripts/    01_fetch_sequences.py, 02_curate_metadata.py, 03_align_and_tree.py\n"
        "    <repo>/config/     host_groups.tsv, vaccine_strains.txt\n"
        "    <repo>/notebooks/  this notebook\n\n"
        "Easiest fix: upload cdv-phylodynamics.zip and run in a cell:\n"
        "    import zipfile, pathlib\n"
        "    zipfile.ZipFile('cdv-phylodynamics.zip').extractall(pathlib.Path.home())"
    )

ROOT = find_repo_root(REPO_ROOT)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "scripts"))
print("repo root:", ROOT)
print("scripts  :", sorted(p.name for p in (ROOT / "scripts").glob("*.py")))

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.grid": True,
    "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False,
})

RAW, INTERIM, PROCESSED = Path("data/raw"), Path("data/interim"), Path("data/processed")

In [ ]:
# Import the curation module so we can reuse its parsers interactively.
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

curate = load_module("curate", ROOT / "scripts" / "02_curate_metadata.py")

# Sanity check that the date parser behaves — cheap insurance against a silent regression
for raw in ["2013", "Aug-2013", "14-Aug-2013", "2013-08-14", "2011/2013", "unknown"]:
    iso, dec, prec = curate.parse_collection_date(raw)
    print(f"{raw:<14} -> {str(dec):<12} ({prec})")

## Step 1 — Fetch from GenBank

Needs your email; the API key is read from the `NCBI_API_KEY` environment variable.
Start with a dry run to see how many records match before downloading anything.

In [ ]:
# Your email. NCBI requires it and will contact you before blocking. Fine to keep here.
EMAIL = "<put your NCBI-registered email here>"          # <-- put your address here

# API key: read from the environment if present, otherwise prompt.
# Either way it is never written into this notebook file.
# On JupyterHub, environment variables usually aren't set, so you'll get a prompt.
# The key lives in memory for this kernel session only — re-enter after a restart.
import getpass

API_KEY = os.environ.get("NCBI_API_KEY", "")
if not API_KEY:
    API_KEY = getpass.getpass("NCBI API key (leave blank to skip): ").strip()
    if API_KEY:
        os.environ["NCBI_API_KEY"] = API_KEY   # so the subprocess scripts see it

assert EMAIL and EMAIL != "you@example.com", "Set EMAIL above before fetching."
print("email  :", EMAIL)
print("api key:", f"set ({len(API_KEY)} chars), 10 req/s" if API_KEY
      else "not set, 3 req/s")


def run_script(script, *args):
    cmd = [sys.executable, f"scripts/{script}", *map(str, args)]
    print("$", " ".join(cmd), "\n")
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print(proc.stdout or proc.stderr)
    return proc.returncode


run_script("01_fetch_sequences.py", "--email", EMAIL, "--dry-run")

In [ ]:
run_script("10_add_entropy.py",
           "--json", "auspice/cdv-phylodynamics.json",
           "--alignment", "data/processed/H_clade_3.fasta",
           "--gene", "H",
           "--output", "auspice/cdv-phylodynamics.json")

In [ ]:
import os, subprocess, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "scripts").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("repo root:", ROOT)

# use the conda env's python if this kernel is base
py = sys.executable
if "cdv-phylo" not in py:
    for r in [Path.home()/"anaconda3", Path.home()/"miniforge3", Path.home()/"miniconda3"]:
        hits = sorted(r.glob("envs/cdv-phylo/bin/python"))
        if hits:
            py = str(hits[0]); break

JSON = ROOT / "auspice/cdv-phylodynamics.json"
ALN  = ROOT / "data/processed/H_clade_3.fasta"

if not ALN.is_file():
    print("\nFASTA files available:")
    for f in sorted((ROOT/"data/processed").glob("*.fasta")):
        print("  ", f.name)
    raise FileNotFoundError("set ALN to the alignment your tree was built from")

cmd = [py, "scripts/10_add_entropy.py",
       "--json", str(JSON.relative_to(ROOT)),
       "--alignment", str(ALN.relative_to(ROOT)),
       "--gene", "H",
       "--output", str(JSON.relative_to(ROOT))]
print("$ " + " ".join(cmd) + "\n")
p = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
print(p.stdout or "")
if p.returncode:
    print("STDERR:", p.stderr[-2000:])

In [ ]:
# Full download. Skips if today's file already exists.
run_script("01_fetch_sequences.py", "--email", EMAIL)

gb_files = sorted(RAW.glob("cdv_*.gb"))
print("\nGenBank files on disk:")
for f in gb_files:
    print(f"  {f}  ({f.stat().st_size/1e6:.1f} MB)")
GB = gb_files[-1] if gb_files else None
GB

## Step 2 — Curate

Parses records, normalizes hosts, flags vaccine strains, parses dates, extracts H sequences.
Re-run this freely — it's deterministic and cheap.

In [ ]:
assert GB is not None, "No GenBank file found. Run the fetch cell first."
run_script("02_curate_metadata.py", "--gb", GB)

In [ ]:
all_df   = pd.read_csv(INTERIM / "metadata_all.tsv", sep="\t")
clean    = pd.read_csv(PROCESSED / "metadata_clean.tsv", sep="\t")
excluded = pd.read_csv(INTERIM / "exclusions.tsv", sep="\t")
review   = pd.read_csv(INTERIM / "needs_review.tsv", sep="\t")

print(f"parsed   {len(all_df):>6}")
print(f"retained {len(clean):>6}  ({len(clean)/len(all_df):.0%})")
print(f"excluded {len(excluded):>6}")
print(f"review   {len(review):>6}")

### Why records were excluded

This table is part of your methods section. Every number here needs a defensible reason.

In [ ]:
(excluded["exclude_reason"].value_counts()
 .rename_axis("reason").reset_index(name="n")
 .assign(pct=lambda d: (100 * d.n / len(all_df)).round(1)))

## Step 3 — The `needs_review` loop

**This is the part that can't be automated.** For each ambiguous record: either add a
pattern to `config/host_groups.tsv`, or accept that it should be dropped.

The helper below lists every host string that failed to match, with counts, so you can
prioritise. A string appearing 40 times is worth a pattern; one appearing once may not be.

In [ ]:
def unmatched_hosts(df):
    """Host strings that didn't match any pattern, most frequent first."""
    u = df[df["host_ambiguous"] & df["host_raw"].notna() & (df["host_raw"].str.strip() != "")]
    return (u["host_raw"].str.strip().value_counts()
            .rename_axis("host_raw").reset_index(name="n"))

unmatched_hosts(all_df).head(40)

In [ ]:
def test_pattern(pattern, df=None):
    """Preview what a candidate pattern would match BEFORE adding it to the config.

    Watch for over-matching: 'dog' matches 'raccoon dog', which is why order
    matters in host_groups.tsv. Check this output for surprises.
    """
    df = all_df if df is None else df
    hits = df[df["host_raw"].fillna("").str.lower().str.contains(pattern.lower(), regex=False)]
    print(f"'{pattern}' matches {len(hits)} records\n")
    return hits["host_raw"].value_counts().rename_axis("host_raw").reset_index(name="n")

test_pattern("fox")

### Ambiguous dates

Dates the parser couldn't read. Some are genuinely unusable (`unknown`, `NA`); others are
formats worth adding to `parse_collection_date()` in the curation script.

In [ ]:
bad_dates = all_df[all_df["decimal_year"].isna() & all_df["collection_date_raw"].notna()]
bad_dates["collection_date_raw"].value_counts().head(25)

### Partial H sequences found by length heuristic

These were identified by sequence length rather than annotation, so they're less certain.
Currently retained and flagged. Decide whether to keep them — more data versus more noise.

In [ ]:
heur = all_df[all_df["H_source"].astype(str).str.startswith("length_heuristic")]
print(f"{len(heur)} records found by heuristic")
heur[["accession", "description", "length", "H_length", "H_source", "host_group"]].head(20)

---
## Step 4 — The Week 3 gate

The decision: is there enough signal across enough hosts to run the analysis as scoped,
or does it need narrowing to felids only, or to one region?

In [ ]:
summary = (clean.groupby("host_group")
           .agg(n=("accession", "size"),
                earliest=("decimal_year", "min"),
                latest=("decimal_year", "max"),
                countries=("country", "nunique"),
                day_precision=("date_precision", lambda s: (s == "day").sum()))
           .sort_values("n", ascending=False))
summary["span_yrs"] = (summary["latest"] - summary["earliest"]).round(1)
summary.round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))

# host group composition
s = clean["host_group"].value_counts()
axes[0,0].barh(s.index[::-1], s.values[::-1], color="#4B3A79")
axes[0,0].set_title("Sequences per host group")
axes[0,0].axvline(5, color="#C0697F", ls="--", lw=1)
axes[0,0].text(5, -0.4, " n=5 floor", color="#C0697F", fontsize=8, va="top")

# sampling through time by host
for grp, sub in clean.groupby("host_group"):
    axes[0,1].scatter(sub["decimal_year"], [grp]*len(sub), s=9, alpha=0.55)
axes[0,1].set_title("Sampling through time")
axes[0,1].set_xlabel("year")

# date precision
p = clean["date_precision"].value_counts()
axes[1,0].bar(p.index, p.values, color="#6E5CA0")
axes[1,0].set_title("Date precision")

# sequences per year
yr = clean["decimal_year"].astype(int).value_counts().sort_index()
axes[1,1].bar(yr.index, yr.values, color="#4B3A79", width=0.85)
axes[1,1].set_title("Sequences per year")

plt.tight_layout()
plt.show()

**How to read these**

- **Host group counts** — the dashed line is a rough floor of 5. Groups below it can't
  support discrete trait analysis and should be merged or dropped.
- **Sampling through time** — you need temporal *spread* within groups, not just totals.
  A wild felid group where every sequence is from 2010–2012 gives BEAST almost nothing.
- **Date precision** — year-only dates are usable but add uncertainty. If most tips are
  year-only, mention it as a limitation.
- **Sequences per year** — heavy recent skew is normal and fine; a gap of a decade is not.

In [ ]:
# The explicit gate check
WILD = ["wild_felid", "wild_canid", "mustelid", "procyonid", "pinniped", "ursid", "ailurid", "viverrid"]
wild = clean[clean["host_group"].isin(WILD)]

print(f"wild carnivore sequences with host + date : {len(wild)}")
print(f"distinct wild host groups (n>=5)          : {(wild['host_group'].value_counts() >= 5).sum()}")
print(f"temporal span of wild sequences           : "
      f"{wild['decimal_year'].min():.1f} - {wild['decimal_year'].max():.1f}")
print(f"domestic dog sequences                    : {(clean['host_group']=='domestic_dog').sum()}")
print()
n = len(wild)
if n >= 150:
    print("=> Comfortable. Proceed with the full cross-host analysis.")
elif n >= 60:
    print("=> Workable. Consider merging the thinner host groups.")
else:
    print("=> Thin. Narrow the scope: felids only, or one geographic region.")
    print("   Send these numbers over and we'll pick the cut together.")

## Step 5 — Hand off to alignment

Once `needs_review` is worked through and the gate passes, relabel and align. The script
deliberately stops after alignment so you look at it before spending compute on a tree.

In [ ]:
run_script("03_align_and_tree.py",
           "--fasta", "data/processed/sequences_H.fasta",
           "--metadata", "data/processed/metadata_clean.tsv",
           "--stop-after", "align")

Open `data/processed/H_aligned.fasta` in **AliView** or **Jalview**. Look for ragged ends,
frame shifts, and any sequence that looks obviously misaligned. Then build the tree:

```bash
python scripts/03_align_and_tree.py \
    --fasta data/processed/sequences_H.fasta \
    --metadata data/processed/metadata_clean.tsv
```

### Then, in order

1. **Open the tree in FigTree.** Does the lineage structure look like published CDV phylogenies?
2. **Find the vaccine clade.** Any "field isolate" sitting inside it is a mislabelled vaccine
   sequence the name filter missed. Add it to `config/vaccine_strains.txt` and re-run from step 2.
   *This check is not optional — name matching provably cannot catch every vaccine-derived sequence.*
3. **Reproduce a published phylogeny** before trusting anything novel.
4. **TempEst** with `H_ml.treefile` and `H_dates.tsv` — check the root-to-tip regression.
   Positive slope and reasonable R² means tip-dated BEAST analysis is viable.

In [ ]:
run_script("03_align_and_tree.py",
           "--fasta", "data/processed/sequences_H.fasta",
           "--metadata", "data/processed/metadata_clean.tsv")

In [ ]:
run_script("05_tree_summary.py",
           "--tree", "data/processed/H_ml.treefile",
           "--write-table")

In [ ]:
run_script("05_tree_summary.py", "--tree", "data/processed/H_ml.treefile",
           "--n-clades", "20", "--write-table")

In [ ]:
# global, stratified to 400
run_script("06_subsample.py", "--mode", "global", "--target", "400",
           "--aln", "data/processed/H_aligned.fasta",
           "--clades", "data/processed/H_ml_clades.tsv",
           "--metadata", "data/processed/metadata_clean.tsv")

# clade_3, the wildlife-maintained lineage — keeps all 204
run_script("06_subsample.py", "--mode", "clade", "--clade", "clade_3",
           "--aln", "data/processed/H_aligned.fasta",
           "--clades", "data/processed/H_ml_clades.tsv",
           "--metadata", "data/processed/metadata_clean.tsv")

---
## Session log

Keep notes here as you go. Decisions you make now will need justifying in the methods
section in four months, and you will not remember why.

In [ ]:
import subprocess, shutil
from pathlib import Path

SUBSETS = ["H_global400", "H_clade_3"]   # add "H_clade_2", "H_clade_5" etc. as you make them
PROC = Path("data/processed")

def read_fasta(p):
    seqs, name, chunks = {}, None, []
    for line in Path(p).read_text().splitlines():
        if line.startswith(">"):
            if name: seqs[name] = "".join(chunks)
            name, chunks = line[1:].split()[0], []
        elif line.strip():
            chunks.append(line.strip())
    if name: seqs[name] = "".join(chunks)
    return seqs

def write_fasta(p, seqs, width=60):
    with Path(p).open("w") as fh:
        for n, s in seqs.items():
            fh.write(f">{n}\n")
            for i in range(0, len(s), width):
                fh.write(s[i:i+width] + "\n")

iqtree = shutil.which("iqtree2") or shutil.which("iqtree")
if not iqtree:
    raise RuntimeError("iqtree not found — conda activate cdv-phylo")

for name in SUBSETS:
    fasta = PROC / f"{name}.fasta"
    if not fasta.is_file():
        print(f"skipping {name} — {fasta} not found\n"); continue

    seqs = read_fasta(fasta)
    L = len(next(iter(seqs.values())))
    keep = [i for i in range(L) if any(s[i] not in "-." for s in seqs.values())]
    clean = PROC / f"{name}_clean.fasta"
    write_fasta(clean, {k: "".join(v[i] for i in keep) for k, v in seqs.items()})
    print(f"{name}: {len(seqs)} seqs, {L} -> {len(keep)} columns "
          f"({L - len(keep)} all-gap removed)")

    cmd = [iqtree, "-s", str(clean), "-m", "MFP", "-B", "1000",
           "--alrt", "1000", "-T", "AUTO",
           "--prefix", str(PROC / f"{name}_ml"), "-redo"]
    print("$", " ".join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print(proc.stdout[-1500:] if proc.returncode == 0 else proc.stdout[-3000:] + proc.stderr[-2000:])
    print(f"--> {PROC / f'{name}_ml.treefile'}\n" + "="*60 + "\n")

In [ ]:
import pandas as pd
from pathlib import Path

PROC = Path("data/processed")
sub = pd.read_csv(PROC / "H_clade_3_metadata.tsv", sep="\t")

# join back to the full metadata for country, strain, description
clean = pd.read_csv(PROC / "metadata_clean.tsv", sep="\t")
clean["acc_base"] = clean["accession"].astype(str).str.split(".").str[0]
cols = [c for c in ["acc_base","description","country","host_raw","host_canonical",
                    "strain","isolate","collection_date"] if c in clean.columns]
d = sub.merge(clean[cols], left_on="accession", right_on="acc_base", how="left")

WILD = {"wild_felid","wild_canid","mustelid","procyonid","pinniped",
        "ursid","ailurid","viverrid"}

print(f"clade_3: {len(d)} sequences\n")
print("HOST COMPOSITION")
h = d["host_group"].value_counts()
for host, n in h.items():
    print(f"  {host:<16} {n:>4}  {n/len(d):>6.1%}{'  <- wild' if host in WILD else ''}")
n_wild = sum(n for k, n in h.items() if k in WILD)
print(f"  {'WILD TOTAL':<16} {n_wild:>4}  {n_wild/len(d):>6.1%}")

print("\nCANONICAL HOST SPECIES (top 15)")
print(d["host_canonical"].value_counts().head(15).to_string())

print("\nGEOGRAPHY")
print(d["country"].value_counts().head(15).to_string())
print(f"  countries: {d['country'].nunique()}  |  missing: {d['country'].isna().sum()}")

print("\nHOST x DECADE")
d["decade"] = (d["decimal_year"] // 10 * 10).astype("Int64")
print(pd.crosstab(d["host_group"], d["decade"]).to_string())

print("\nDATE PRECISION")
print(d["date_precision"].value_counts().to_string())
print(f"  span: {d['decimal_year'].min():.1f} - {d['decimal_year'].max():.1f}")

print("\n" + "="*66)
print("THE 9 DOMESTIC DOGS — the most interesting tips in this clade")
print("="*66)
dogs = d[d["host_group"] == "domestic_dog"]
print(dogs[["accession","country","collection_date","strain","description"]]
      .to_string(index=False, max_colwidth=55))

unresolved = d[d["host_group"].isin(["other","unknown","unparsed"])]
if len(unresolved):
    print(f"\nUNRESOLVED HOSTS ({len(unresolved)}) — check these")
    print(unresolved[["accession","host_raw","description"]].to_string(index=False, max_colwidth=55))

In [ ]:
run_script("07_make_beast_xml.py", "--aln", "data/processed/H_clade_3_clean.fasta",
           "--out", "beast/drt", "--rate", "7.46e-4",
           "--chain", "20000000", "--randomize-dates", "20")

In [ ]:
from pathlib import Path
for d in ["data/processed", "beast", "beast/clade3"]:
    p = Path(d)
    print(f"\n{d}:", "(missing)" if not p.is_dir() else "")
    if p.is_dir():
        for f in sorted(p.iterdir()):
            print(f"   {f.name:<45} {f.stat().st_size/1024:>8.1f} KB")

In [ ]:
# 2026-__-__
# records fetched:
# retained after curation:
# host patterns added:
# records dropped by hand, and why:
# gate decision:

In [ ]:
run_script("07_make_beast_xml.py",
           "--aln", "data/processed/H_clade_3.fasta",
           "--out", "beast/clade3",
           "--rate", "7.46e-4",
           "--chain", "100000000")

In [ ]:
run_script("07_make_beast_xml.py",
           "--aln", "data/processed/H_clade_3.fasta",
           "--out", "beast/clade3",
           "--rate", "7.46e-4",
           "--chain", "100000000")

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

XML = Path("beast/clade3/H_clade_3.xml")   # adjust to your filename
root = ET.parse(XML).getroot()
taxa = [s.get("taxon") for s in root.findall(".//sequence")]

print(f"{len(taxa)} taxa in the XML")
print("example:", repr(taxa[0]))

out = XML.parent
with (out / "traits_with_header.txt").open("w") as fh:
    fh.write("traits\thost\n")
    for t in taxa:
        fh.write(f"{t}\t{t.split('|')[1]}\n")

with (out / "traits_no_header.txt").open("w") as fh:
    for t in taxa:
        fh.write(f"{t}\t{t.split('|')[1]}\n")

print("wrote both variants to", out)
print("first data line:", repr(f"{taxa[0]}\t{taxa[0].split('|')[1]}"))

In [ ]:
import shutil, subprocess, time, re
from pathlib import Path

XML = Path("beast/clade3/H_clade_3.xml")   # <-- your edited test XML
SEED = 12345
THREADS = 4
TIMEOUT_MIN = 30        # kill it if the "test" turns into an all-nighter

def find_beast():
    exe = shutil.which("beast")
    if exe: return exe
    for pat in ["/Applications/BEAST*/bin/beast",
                str(Path.home() / "BEAST*/bin/beast"),
                str(Path.home() / "Applications/BEAST*/bin/beast")]:
        hits = sorted(Path(pat).parent.parent.parent.glob(Path(pat).relative_to(Path(pat).parents[2]).as_posix())) \
               if False else sorted(Path("/").glob(pat.lstrip("/")))
        if hits: return str(hits[-1])
    return None

beast = find_beast()
if not beast:
    raise RuntimeError("BEAST not found. Install from beast2.org, or set `beast` manually "
                       "to the full path of the executable inside BEAST*/bin/")
if not XML.is_file():
    raise FileNotFoundError(f"{XML} not found. Files in {XML.parent}: "
                            f"{sorted(p.name for p in XML.parent.iterdir()) if XML.parent.is_dir() else 'dir missing'}")

# confirm the chain length you actually set
chain = re.search(r'chainLength="(\d+)"', XML.read_text())
print(f"beast : {beast}")
print(f"xml   : {XML}")
print(f"chain : {int(chain.group(1)):,}" if chain else "chain : not found")
print("-" * 60)

cmd = [beast, "-seed", str(SEED), "-threads", str(THREADS),
       "-overwrite", "-working", str(XML)]
t0 = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
posteriors, tail = [], []
try:
    for line in proc.stdout:
        line = line.rstrip()
        tail.append(line); tail[:] = tail[-40:]
        parts = line.split()
        if len(parts) >= 2 and parts[0].isdigit():
            try:
                posteriors.append(float(parts[1]))
            except ValueError:
                pass
            if len(posteriors) % 10 == 1:
                print(f"  [{time.time()-t0:6.0f}s] {line[:90]}")
        elif any(k in line for k in ("Error", "Exception", "error", "Fatal")):
            print("  !!", line)
        if time.time() - t0 > TIMEOUT_MIN * 60:
            proc.kill(); print(f"\nKILLED after {TIMEOUT_MIN} min"); break
finally:
    proc.wait()

print("-" * 60)
print(f"exit code {proc.returncode} after {(time.time()-t0)/60:.1f} min")
if proc.returncode != 0:
    print("\nLast lines:\n  " + "\n  ".join(tail[-15:]))
elif posteriors:
    print(f"posterior: {posteriors[0]:.1f} -> {posteriors[-1]:.1f} "
          f"({len(posteriors)} samples)")
    print("climbing then plateauing is what you want" if posteriors[-1] > posteriors[0]
          else "posterior did not improve — check the priors")

for f in sorted(XML.parent.glob("*.log")) + sorted(XML.parent.glob("*.trees")):
    print(f"  {f.name:<40} {f.stat().st_size/1024:>9.1f} KB")

In [ ]:
from pathlib import Path
import re

d = Path("beast/clade3")   # adjust to where your XML lives
logs = sorted(d.glob("*.log"))
if not logs:
    print("No .log file — BEAST likely never got past initialisation.")
else:
    f = max(logs, key=lambda p: p.stat().st_mtime)
    lines = [l for l in f.read_text().splitlines() if l and not l.startswith("#")]
    data = [l for l in lines[1:] if l.split()[0].isdigit()]
    print(f"{f.name}: {len(data)} samples logged")
    if data:
        last_state = int(data[-1].split()[0])
        mins = 30
        print(f"reached state {last_state:,} in ~{mins} min")
        print(f"rate: {last_state/mins*60:,.0f} states/hour")
        print(f"\n100M states would take {100_000_000/(last_state/mins*60):,.0f} hours "
              f"({100_000_000/(last_state/mins*60)/24:,.1f} days)")
        post = [float(l.split()[1]) for l in data]
        print(f"posterior: {post[0]:.1f} -> {post[-1]:.1f}")
        print("climbing — the chain was working, just slow" if post[-1] > post[0]
              else "flat/erratic — check priors before blaming speed")

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

XML = Path("beast/clade3/H_clade_3.xml")   # <-- the file you actually ran

print("="*66); print(f"  {XML}"); print("="*66)
if not XML.is_file():
    d = XML.parent
    raise FileNotFoundError(f"not found. In {d}: "
        f"{sorted(p.name for p in d.iterdir()) if d.is_dir() else 'dir missing'}")

raw = XML.read_text()
root = ET.fromstring(raw)

for r in root.findall(".//run"):
    cl = r.get("chainLength")
    print(f"chainLength : {int(cl):,}" if cl and cl.isdigit() else f"chainLength : {cl}")

data = root.findall(".//data")
print(f"\ndata partitions : {len(data)}")
for d in data:
    print(f"  id={d.get('id')!r:<30} sequences={len(d.findall('./sequence'))}")

print(f"\ntraitsets : {len(root.findall('.//trait'))}")
for t in root.findall(".//trait"):
    val = t.get("value") or ""
    n = len([v for v in val.split(",") if v.strip()])
    print(f"  traitname={t.get('traitname')!r:<18} entries={n:<5} "
          f"e.g. {val.split(',')[0].strip()[:45]}")

print("\nDISCRETE TRAIT CHECK")
print(f"  'host' appears               : {'host' in raw}")
print(f"  AncestralStateTreeLikelihood : {'AncestralState' in raw}")
print(f"  SVSGeneralSubstitutionModel  : {'SVSGeneralSubstitutionModel' in raw}")
print(f"  BSSVS indicators             : {'indicator' in raw.lower()}")

print("\nLOGGERS")
for lg in root.findall(".//logger"):
    print(f"  {str(lg.get('fileName')):<34} every={lg.get('logEvery'):<8} "
          f"mode={lg.get('mode') or 'params'}")

print("\nVERDICT")
has_dta = len(data) > 1 or "AncestralState" in raw or "SVSGeneralSubstitutionModel" in raw
has_hostlog = any("host" in (lg.get("fileName") or "") for lg in root.findall(".//logger"))
if not has_dta:
    print("  !! NO DISCRETE TRAIT PARTITION — you'd get a timed tree but no")
    print("     host-jump reconstruction, which is the actual result you want.")
if not has_hostlog:
    print("  !! no host.trees logger — ancestral states wouldn't be saved")
if has_dta and has_hostlog:
    print("  complete DTA analysis — ready to run")

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

XML = Path("beast/clade3/clade3_dta.xml")
raw = XML.read_text()
root = ET.fromstring(raw)

print(f"data partitions : {len(root.findall('.//data'))}")
for r in root.findall(".//run"):
    cl = r.get("chainLength")
    print(f"chainLength     : {int(cl):,}" if cl and cl.isdigit() else f"chainLength : {cl}")
print(f"BSSVS indicators: {'indicator' in raw.lower()}")
print(f"asymmetric model: {'SVSGeneralSubstitutionModel' in raw or 'AncestralState' in raw}")
print("\nloggers:")
for lg in root.findall(".//logger"):
    print(f"  {str(lg.get('fileName')):<34} every={lg.get('logEvery')}")

In [ ]:
from pathlib import Path
import time

hits = [p for p in Path.home().rglob("*.xml")
        if "dta" in p.name.lower() or "clade3" in p.name.lower()]
hits.sort(key=lambda p: p.stat().st_mtime, reverse=True)

for p in hits[:10]:
    age = (time.time() - p.stat().st_mtime) / 60
    print(f"{age:7.1f} min ago  {p.stat().st_size/1024:8.1f} KB  {p}")

In [ ]:
import shutil
src = Path(Users/batyanightingale/projects/cdv-phylodynamics/beast/clade3_dta.xml)
dst = Path(Users/batyanightingale/projects/cdv-phylodynamics/beast/clade3/clade3_dta.xml)
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.move(str(src), dst)
print("now at", dst.resolve())

In [ ]:
import shutil
from pathlib import Path

src = Path("$HOME/projects/cdv-phylodynamics/beast/clade3_dta.xml")
dst = Path("$HOME/projects/cdv-phylodynamics/beast/clade3/clade3_dta.xml")
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.move(str(src), str(dst))
print("now at", dst.resolve())

In [ ]:
XML = Path("beast/clade3/clade3_dta.xml")


In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

XML = Path("beast/clade3/clade3_dta.xml")
raw = XML.read_text()
root = ET.fromstring(raw)

data = root.findall(".//data")
print(f"data partitions : {len(data)}")
for d in data:
    print(f"    id={d.get('id')!r:<28} sequences={len(d.findall('./sequence'))}")

for r in root.findall(".//run"):
    cl = r.get("chainLength")
    print(f"chainLength     : {int(cl):,}" if cl and cl.isdigit() else f"chainLength : {cl}")

print(f"BSSVS indicators: {'indicator' in raw.lower()}")
print(f"trait model     : {'SVSGeneralSubstitutionModel' in raw or 'AncestralState' in raw}")

print("\nloggers:")
for lg in root.findall(".//logger"):
    print(f"    {str(lg.get('fileName')):<34} every={lg.get('logEvery')}")

ok = len(data) >= 2 and any("host" in (lg.get("fileName") or "") for lg in root.findall(".//logger"))
print("\n" + ("READY — trait partition and host logger both present" if ok
              else "NOT READY — see above"))

In [ ]:
import re, shutil
import xml.etree.ElementTree as ET
from pathlib import Path

XML       = Path("beast/clade3/clade3_dta.xml")
CHAIN     = 10_000_000    # keep 10M; raise later only if ESS falls short
LOG_EVERY = 10_000

backup = XML.with_suffix(".xml.bak")
shutil.copy(XML, backup)
print(f"backup -> {backup.name}\n")

raw = XML.read_text()

raw, n_chain = re.subn(r'(<run\b[^>]*?chainLength=")\d+(")',
                       rf'\g<1>{CHAIN}\g<2>', raw)

def fix_logger(m):
    tag = m.group(0)
    if 'id="screenlog"' in tag:        # leave console output alone
        return tag
    return re.sub(r'logEvery="\d+"', f'logEvery="{LOG_EVERY}"', tag)

raw, n_log = re.subn(r'<logger\b[^>]*>', fix_logger, raw)
XML.write_text(raw)

# verify
root = ET.fromstring(XML.read_text())
for r in root.findall(".//run"):
    print(f"chainLength : {int(r.get('chainLength')):,}")
print("\nloggers:")
for lg in root.findall(".//logger"):
    print(f"  {str(lg.get('fileName')):<34} every={lg.get('logEvery')}")

samples = CHAIN // LOG_EVERY
print(f"\n{samples} samples per file — {'good' if samples >= 500 else 'thin, lower LOG_EVERY'}")
print(f"chain edits: {n_chain} | loggers scanned: {n_log}")

In [ ]:
import shutil, subprocess, time, re
from pathlib import Path

XML         = Path("beast/clade3/clade3_dta.xml")
SEED        = 12345
THREADS     = 4
TIMEOUT_MIN = 90

def find_beast():
    exe = shutil.which("beast")
    if exe:
        return exe
    for root in [Path("/Applications"), Path.home() / "Applications", Path.home()]:
        if root.is_dir():
            for pattern in ("BEAST*/bin/beast", "*/BEAST*/bin/beast"):
                hits = sorted(root.glob(pattern))
                if hits:
                    return str(hits[0])
    return None

beast = find_beast()
if not beast:
    raise RuntimeError("BEAST not found. Set `beast` to the full path of BEAST*/bin/beast")
if not XML.is_file():
    d = XML.parent
    raise FileNotFoundError(f"{XML} missing. In {d}: "
        f"{sorted(p.name for p in d.iterdir()) if d.is_dir() else 'dir missing'}")

chain = re.search(r'chainLength="(\d+)"', XML.read_text())
print(f"beast : {beast}")
print(f"xml   : {XML}")
print(f"chain : {int(chain.group(1)):,}" if chain else "chain : ?")
print(f"seed  : {SEED}")
print("-"*66)

cmd = [beast, "-seed", str(SEED), "-threads", str(THREADS), "-overwrite", str(XML)]
t0 = time.time()
proc = subprocess.Popen(cmd, cwd=XML.parent, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
posteriors, tail, n = [], [], 0
try:
    for line in proc.stdout:
        line = line.rstrip(); tail.append(line); tail[:] = tail[-40:]
        low = line.lower()
        if any(k in low for k in ("state", "trait", "beagle", "error", "exception")) and n < 25:
            print(f"  {line[:100]}"); n += 1
        parts = line.split()
        if len(parts) >= 2 and parts[0].isdigit():
            try:
                posteriors.append(float(parts[1]))
                if len(posteriors) % 25 == 1:
                    el = time.time() - t0
                    frac = int(parts[0]) / int(chain.group(1)) if chain else 0
                    eta = (el/frac - el)/60 if frac > 0.01 else 0
                    print(f"  [{el/60:5.1f} min] state {int(parts[0]):>12,}  "
                          f"posterior {parts[1]:>12}  ETA {eta:5.1f} min")
            except ValueError:
                pass
        if time.time() - t0 > TIMEOUT_MIN*60:
            proc.kill(); print(f"\nKILLED at {TIMEOUT_MIN} min"); break
finally:
    proc.wait()

print("-"*66)
print(f"exit {proc.returncode} after {(time.time()-t0)/60:.1f} min")
if proc.returncode != 0:
    print("\nlast lines:\n  " + "\n  ".join(tail[-15:]))
elif posteriors:
    print(f"posterior {posteriors[0]:,.1f} -> {posteriors[-1]:,.1f}")

print("\noutput files:")
for f in sorted(XML.parent.glob("*.log")) + sorted(XML.parent.glob("*.trees")):
    kb = f.stat().st_size/1024
    flag = "  <-- EMPTY" if kb < 1 else ""
    print(f"  {f.name:<38} {kb:>10.1f} KB{flag}")

In [ ]:
import shutil, subprocess, time, re
from pathlib import Path

XML         = Path("beast/clade3/clade3_dta.xml").resolve()   # absolute: fixes the doubled path
SEED        = 12345
THREADS     = 4
TIMEOUT_MIN = 90

# --- move the old sequence-only outputs aside ---
for f in ["H_clade_3.log", "H_clade_3.trees"]:
    p = XML.parent / f
    if p.exists():
        p.rename(XML.parent / ("OLD_seqonly_" + f))
        print(f"renamed {f} -> OLD_seqonly_{f}")

def find_beast():
    exe = shutil.which("beast")
    if exe:
        return exe
    for root in [Path("/Applications"), Path.home() / "Applications", Path.home()]:
        if root.is_dir():
            for pattern in ("BEAST*/bin/beast", "*/BEAST*/bin/beast"):
                hits = sorted(root.glob(pattern))
                if hits:
                    return str(hits[0])
    return None

beast = find_beast()
if not beast:
    raise RuntimeError("BEAST not found — set `beast` to the full path of BEAST*/bin/beast")
if not XML.is_file():
    raise FileNotFoundError(f"{XML} missing")

chain = re.search(r'chainLength="(\d+)"', XML.read_text())
total = int(chain.group(1)) if chain else None
print(f"\nbeast : {beast}")
print(f"xml   : {XML}")
print(f"chain : {total:,}" if total else "chain : ?")
print(f"seed  : {SEED}")
print("-"*66)

cmd = [beast, "-seed", str(SEED), "-threads", str(THREADS), "-overwrite", str(XML)]
t0 = time.time()
proc = subprocess.Popen(cmd, cwd=str(XML.parent), stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
posteriors, tail, shown = [], [], 0
try:
    for line in proc.stdout:
        line = line.rstrip(); tail.append(line); tail[:] = tail[-40:]
        low = line.lower()
        if any(k in low for k in ("trait", "beagle", "error", "exception", "states")) and shown < 25:
            print(f"  {line[:100]}"); shown += 1
        parts = line.split()
        if len(parts) >= 2 and parts[0].isdigit():
            try:
                posteriors.append(float(parts[1]))
                if len(posteriors) % 25 == 1 and total:
                    el = time.time() - t0
                    frac = int(parts[0]) / total
                    eta = (el/frac - el)/60 if frac > 0.01 else 0
                    print(f"  [{el/60:5.1f} min] state {int(parts[0]):>12,}  "
                          f"posterior {parts[1]:>12}  ETA {eta:5.1f} min")
            except ValueError:
                pass
        if time.time() - t0 > TIMEOUT_MIN*60:
            proc.kill(); print(f"\nKILLED at {TIMEOUT_MIN} min"); break
finally:
    proc.wait()

print("-"*66)
print(f"exit {proc.returncode} after {(time.time()-t0)/60:.1f} min")
if proc.returncode != 0:
    print("\nlast lines:\n  " + "\n  ".join(tail[-15:]))
elif posteriors:
    print(f"posterior {posteriors[0]:,.1f} -> {posteriors[-1]:,.1f}")

print("\noutput files:")
for f in sorted(XML.parent.glob("clade3_dta*")) + sorted(XML.parent.glob("Host*")):
    kb = f.stat().st_size/1024
    print(f"  {f.name:<40} {kb:>10.1f} KB" + ("   <-- EMPTY" if kb < 1 else ""))

In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path
import shutil

XML   = Path("beast/clade3/clade3_dta.xml").resolve()
OUT   = XML.parent / "clade3_dta_merged.xml"
REMAP = {"ailurid": "wild_felid"}      # single-sequence state folded in

raw  = XML.read_text()
root = ET.fromstring(raw)

for el in root.iter():
    tn, val = el.get("traitname"), el.get("value")
    if not val or not tn or tn == "date-forward":
        continue
    pairs = [p for p in val.split(",") if "=" in p]
    print(f"traitname={tn!r} — {len(pairs)} taxa")
    print("  before:", dict(Counter(p.rsplit('=',1)[1].strip() for p in pairs)))
    new_pairs = [f"{p.rsplit('=',1)[0]}={REMAP.get(p.rsplit('=',1)[1].strip(), p.rsplit('=',1)[1].strip())}"
                 for p in pairs]
    print("  after :", dict(Counter(p.rsplit('=',1)[1] for p in new_pairs)))
    raw = raw.replace(val, ",".join(new_pairs))

ET.fromstring(raw)          # fails loudly if the edit broke anything
OUT.write_text(raw)
print(f"\nwrote {OUT.name}")

In [ ]:
import shutil, subprocess, time, re
from pathlib import Path

XML         = Path("beast/clade3/clade3_dta_merged.xml").resolve()
SEED        = 12345
THREADS     = 4
TIMEOUT_MIN = 120

# park the outputs from the crashed run — Host_tree_with_trait.trees is corrupt
crashed = XML.parent / "crashed_run"
moved = []
for f in XML.parent.glob("*"):
    if f.is_file() and (f.name.startswith("clade3_dta.") and f.suffix in (".log", ".trees")
                        or f.name.startswith("Host_tree_with_trait")):
        crashed.mkdir(exist_ok=True)
        shutil.move(str(f), crashed / f.name)
        moved.append(f.name)
print(f"moved to crashed_run/: {moved}" if moved else "no previous outputs to move")

def find_beast():
    exe = shutil.which("beast")
    if exe:
        return exe
    for root in [Path("/Applications"), Path.home() / "Applications", Path.home()]:
        if root.is_dir():
            for pattern in ("BEAST*/bin/beast", "*/BEAST*/bin/beast"):
                hits = sorted(root.glob(pattern))
                if hits:
                    return str(hits[0])
    return None

beast = find_beast()
if not beast:
    raise RuntimeError("BEAST not found")
if not XML.is_file():
    raise FileNotFoundError(f"{XML} missing — run the remap cell first")

chain = re.search(r'chainLength="(\d+)"', XML.read_text())
total = int(chain.group(1)) if chain else None
print(f"\nbeast : {beast}")
print(f"xml   : {XML.name}")
print(f"chain : {total:,}" if total else "chain : ?")
print("-"*66)

cmd = [beast, "-seed", str(SEED), "-threads", str(THREADS), "-overwrite", str(XML)]
t0 = time.time()
proc = subprocess.Popen(cmd, cwd=str(XML.parent), stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
posteriors, tail, shown, errors = [], [], 0, 0
try:
    for line in proc.stdout:
        line = line.rstrip(); tail.append(line); tail[:] = tail[-40:]
        low = line.lower()
        if "randomchoiceunnormalized" in low or "java.lang.Error" in line:
            errors += 1
            if errors <= 3:
                print(f"  !! {line[:100]}")
            if errors == 4:
                print("  !! still crashing — kill this and switch to a symmetric trait model")
        elif any(k in low for k in ("trait", "beagle", "states", "writing file")) and shown < 20:
            print(f"  {line[:100]}"); shown += 1
        parts = line.split()
        if len(parts) >= 2 and parts[0].isdigit():
            try:
                posteriors.append(float(parts[1]))
                if len(posteriors) % 25 == 1 and total:
                    el = time.time() - t0
                    frac = int(parts[0]) / total
                    eta = (el/frac - el)/60 if frac > 0.01 else 0
                    print(f"  [{el/60:5.1f} min] state {int(parts[0]):>12,}  "
                          f"posterior {parts[1]:>12}  ETA {eta:5.1f} min")
            except ValueError:
                pass
        if time.time() - t0 > TIMEOUT_MIN*60:
            proc.kill(); print(f"\nKILLED at {TIMEOUT_MIN} min"); break
finally:
    proc.wait()

print("-"*66)
print(f"exit {proc.returncode} after {(time.time()-t0)/60:.1f} min | trait errors: {errors}")
if proc.returncode != 0:
    print("\nlast lines:\n  " + "\n  ".join(tail[-12:]))
elif posteriors:
    print(f"posterior {posteriors[0]:,.1f} -> {posteriors[-1]:,.1f}")

print("\noutput files:")
for f in sorted(XML.parent.glob("*.log")) + sorted(XML.parent.glob("*.trees")):
    kb = f.stat().st_size/1024
    print(f"  {f.name:<40} {kb:>10.1f} KB" + ("   <-- EMPTY" if kb < 1 else ""))

In [ ]:
from pathlib import Path
import hashlib, re
from collections import Counter

d = Path("beast/clade3")
for name in ["clade3_dta.xml", "clade3_dta_merged.xml"]:
    p = d / name
    raw = p.read_text()
    print(f"{name}: {hashlib.md5(raw.encode()).hexdigest()[:12]}  {len(raw):,} chars")
    print("   ailurid occurrences:", raw.count("ailurid"))

# where do the trait states actually live?
raw = (d / "clade3_dta.xml").read_text()
for m in re.finditer(r'<(\w+)[^>]*traitname="([^"]+)"', raw):
    print(f"\nelement <{m.group(1)}> traitname={m.group(2)!r}")

In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

XML   = Path("beast/clade3/clade3_dta.xml").resolve()
OUT   = XML.parent / "clade3_dta_merged.xml"
REMAP = {"ailurid": "wild_felid"}

tree = ET.parse(XML); root = tree.getroot()
changed = 0
for el in root.iter():
    tn = el.get("traitname")
    if not tn or tn.startswith("date"):
        continue
    src = "attr" if el.get("value") else "text"
    val = el.get("value") or (el.text or "")
    pairs = [x.strip() for x in val.replace("\n", " ").split(",") if "=" in x]
    if not pairs:
        continue
    print(f"<{el.tag}> traitname={tn!r} source={src} taxa={len(pairs)}")
    print("  before:", dict(Counter(x.rsplit('=',1)[1].strip() for x in pairs)))
    new = [f"{x.rsplit('=',1)[0].strip()}="
           f"{REMAP.get(x.rsplit('=',1)[1].strip(), x.rsplit('=',1)[1].strip())}" for x in pairs]
    print("  after :", dict(Counter(x.rsplit('=',1)[1] for x in new)))
    if src == "attr":
        el.set("value", ",".join(new))
    else:
        el.text = "\n" + ",\n".join(new) + "\n"
    changed += 1

if not changed:
    raise RuntimeError("no discrete traitset found — paste me the <traitSet> block")
tree.write(OUT, encoding="unicode", xml_declaration=True)
print(f"\nchanged {changed} element(s) -> {OUT.name}")
print("ailurid remaining:", OUT.read_text().count("=ailurid"))

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
p = Path("beast/clade3/clade3_dta_merged.xml")
r = ET.fromstring(p.read_text())
print("data partitions:", len(r.findall(".//data")))
print("chainLength   :", [x.get("chainLength") for x in r.findall(".//run")])
print("loggers       :", [x.get("fileName") for x in r.findall(".//logger")])
print("BSSVS         :", "indicator" in p.read_text().lower())

In [ ]:
XML = Path("beast/clade3/clade3_dta_merged.xml").resolve()

In [ ]:
import shutil, subprocess, time, re
from pathlib import Path

XML         = Path("beast/clade3/clade3_dta_merged.xml").resolve()
SEED        = 12345
THREADS     = 4
TIMEOUT_MIN = 120

# park outputs from the previous crashed attempt
crashed = XML.parent / "crashed_run2"
moved = []
for f in XML.parent.glob("*"):
    if f.is_file() and (f.name.startswith("clade3_dta.") and f.suffix in (".log", ".trees")
                        or f.name.startswith("Host_tree_with_trait")):
        crashed.mkdir(exist_ok=True)
        shutil.move(str(f), crashed / f.name)
        moved.append(f.name)
print(f"moved aside: {moved}" if moved else "no previous outputs to move")

def find_beast():
    exe = shutil.which("beast")
    if exe:
        return exe
    for root in [Path("/Applications"), Path.home() / "Applications", Path.home()]:
        if root.is_dir():
            for pattern in ("BEAST*/bin/beast", "*/BEAST*/bin/beast"):
                hits = sorted(root.glob(pattern))
                if hits:
                    return str(hits[0])
    return None

beast = find_beast()
if not beast:
    raise RuntimeError("BEAST not found")
if not XML.is_file():
    raise FileNotFoundError(f"{XML} missing — run the remap cell first")

chain = re.search(r'chainLength="(\d+)"', XML.read_text())
total = int(chain.group(1)) if chain else None
print(f"\nbeast : {beast}")
print(f"xml   : {XML.name}")
print(f"chain : {total:,}" if total else "chain : ?")
print("-"*66)

cmd = [beast, "-seed", str(SEED), "-threads", str(THREADS), "-overwrite", str(XML)]
t0 = time.time()
proc = subprocess.Popen(cmd, cwd=str(XML.parent), stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
posteriors, tail, shown, errors = [], [], 0, 0
passed_500k = False
try:
    for line in proc.stdout:
        line = line.rstrip(); tail.append(line); tail[:] = tail[-40:]
        low = line.lower()
        if "randomchoiceunnormalized" in low:
            errors += 1
            if errors <= 3:
                print(f"  !! trait error {errors} — {line[:80]}")
            if errors == 4:
                print("  !! SAME CRASH — kill this. Untick BSSVS in BEAUti, set Symmetric.")
        elif any(k in low for k in ("trait", "beagle", "states", "writing file")) and shown < 18:
            print(f"  {line[:100]}"); shown += 1
        parts = line.split()
        if len(parts) >= 2 and parts[0].isdigit():
            state = int(parts[0])
            if state > 500_000 and not passed_500k and errors == 0:
                passed_500k = True
                print("  >> past 500,000 states with no trait errors — the merge fixed it")
            try:
                posteriors.append(float(parts[1]))
                if len(posteriors) % 25 == 1 and total:
                    el = time.time() - t0
                    frac = state / total
                    eta = (el/frac - el)/60 if frac > 0.01 else 0
                    print(f"  [{el/60:5.1f} min] state {state:>12,}  "
                          f"posterior {parts[1]:>12}  ETA {eta:5.1f} min")
            except ValueError:
                pass
        if time.time() - t0 > TIMEOUT_MIN*60:
            proc.kill(); print(f"\nKILLED at {TIMEOUT_MIN} min"); break
finally:
    proc.wait()

print("-"*66)
print(f"exit {proc.returncode} after {(time.time()-t0)/60:.1f} min | trait errors: {errors}")
if proc.returncode != 0:
    print("\nlast lines:\n  " + "\n  ".join(tail[-12:]))
elif posteriors:
    print(f"posterior {posteriors[0]:,.1f} -> {posteriors[-1]:,.1f}")

print("\noutput files:")
for f in sorted(XML.parent.glob("*.log")) + sorted(XML.parent.glob("*.trees")):
    kb = f.stat().st_size/1024
    print(f"  {f.name:<40} {kb:>10.1f} KB" + ("   <-- EMPTY" if kb < 1 else ""))

In [ ]:
import shutil, subprocess, time, re
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

SRC         = Path("beast/clade3/clade3_dta_nobssvs.xml").resolve()
FINAL       = SRC.parent / "clade3_dta_final.xml"
SEEDS       = [12345, 54321]
THREADS     = 4
TIMEOUT_MIN = 120
REMAP       = {"ailurid": "wild_felid"}

# ---------- 1. merge the single-sequence state ----------
tree = ET.parse(SRC); root = tree.getroot()
changed = 0
for el in root.iter():
    tn = el.get("traitname")
    if not tn or tn.startswith("date"):
        continue
    src = "attr" if el.get("value") else "text"
    val = el.get("value") or (el.text or "")
    pairs = [y.strip() for y in val.replace("\n", " ").split(",") if "=" in y]
    if not pairs:
        continue
    print(f"<{el.tag}> traitname={tn!r} taxa={len(pairs)}")
    print("  before:", dict(Counter(y.rsplit('=',1)[1].strip() for y in pairs)))
    new = [f"{y.rsplit('=',1)[0].strip()}="
           f"{REMAP.get(y.rsplit('=',1)[1].strip(), y.rsplit('=',1)[1].strip())}" for y in pairs]
    print("  after :", dict(Counter(y.rsplit('=',1)[1] for y in new)))
    if src == "attr":
        el.set("value", ",".join(new))
    else:
        el.text = "\n" + ",\n".join(new) + "\n"
    changed += 1
if not changed:
    raise RuntimeError("no discrete traitset found")
tree.write(FINAL, encoding="unicode", xml_declaration=True)

# ---------- 2. verify ----------
raw = FINAL.read_text(); r = ET.fromstring(raw)
chain = re.search(r'chainLength="(\d+)"', raw)
total = int(chain.group(1)) if chain else None
print(f"\n{FINAL.name}")
print(f"  partitions : {len(r.findall('.//data'))}")
print(f"  chainLength: {total:,}" if total else "  chainLength: ?")
print(f"  BSSVS off  : {'indicator' not in raw.lower()}")
print(f"  ailurid    : {raw.count('=ailurid')} remaining")
print(f"  loggers    : {[x.get('fileName') for x in r.findall('.//logger') if x.get('fileName')]}")

# ---------- 3. park old outputs ----------
old = SRC.parent / "previous_runs"
for f in list(SRC.parent.glob("clade3_dta.*")) + list(SRC.parent.glob("Host_tree_with_trait*")):
    if f.is_file() and f.suffix in (".log", ".trees"):
        old.mkdir(exist_ok=True); shutil.move(str(f), old / f.name)

def find_beast():
    exe = shutil.which("beast")
    if exe: return exe
    for rt in [Path("/Applications"), Path.home()/"Applications", Path.home()]:
        if rt.is_dir():
            for pat in ("BEAST*/bin/beast", "*/BEAST*/bin/beast"):
                hits = sorted(rt.glob(pat))
                if hits: return str(hits[0])
    return None

beast = find_beast()
if not beast:
    raise RuntimeError("BEAST not found")

# ---------- 4. run both chains ----------
for seed in SEEDS:
    print("\n" + "="*66); print(f"  CHAIN seed={seed}"); print("="*66)
    t0 = time.time(); errors = 0; posteriors = []; tail = []; ok500 = False
    proc = subprocess.Popen([beast, "-seed", str(seed), "-threads", str(THREADS),
                             "-overwrite", str(FINAL)],
                            cwd=str(FINAL.parent), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout:
            line = line.rstrip(); tail.append(line); tail[:] = tail[-30:]
            if "randomchoiceunnormalized" in line.lower():
                errors += 1
                if errors <= 2: print(f"  !! trait error {errors}")
            parts = line.split()
            if len(parts) >= 2 and parts[0].isdigit():
                state = int(parts[0])
                if state > 500_000 and not ok500 and errors == 0:
                    ok500 = True; print("  >> 500k states clean")
                try:
                    posteriors.append(float(parts[1]))
                    if len(posteriors) % 50 == 1 and total:
                        el = time.time()-t0; frac = state/total
                        eta = (el/frac - el)/60 if frac > 0.01 else 0
                        print(f"  [{el/60:5.1f} min] {state:>12,}  {parts[1]:>12}  ETA {eta:5.1f} min")
                except ValueError:
                    pass
            if time.time()-t0 > TIMEOUT_MIN*60:
                proc.kill(); print("  KILLED on timeout"); break
    finally:
        proc.wait()

    print(f"  exit {proc.returncode} in {(time.time()-t0)/60:.1f} min | trait errors: {errors}")
    if proc.returncode != 0:
        print("  last lines:\n    " + "\n    ".join(tail[-8:])); break

    for f in ["clade3_dta.log", "clade3_dta.trees", "Host_tree_with_trait.trees"]:
        p = FINAL.parent / f
        if p.exists():
            p.rename(FINAL.parent / f.replace(".", f"_seed{seed}.", 1))

print("\nfinal outputs:")
for f in sorted(FINAL.parent.glob("*seed*")):
    print(f"  {f.name:<48} {f.stat().st_size/1024:>10.1f} KB")

In [ ]:
from pathlib import Path
import re
raw = Path("beast/clade3/clade3_dta_nobssvs.xml").read_text()
for kw in ["indicator", "SVSGeneralSubstitutionModel", "GeneralSubstitutionModel",
           "Symmetric", "Asymmetric", "BSSVS", "nonZeroRates"]:
    print(f"  {kw:<32} {raw.lower().count(kw.lower())}")
print("\nlines mentioning indicator or SubstitutionModel:")
for l in raw.splitlines():
    if re.search(r"indicator|SubstitutionModel", l, re.I):
        print("   ", l.strip()[:120])

In [ ]:
from pathlib import Path
from collections import Counter

src = Path("data/processed/H_clade_3.fasta")
out = Path("data/processed/H_clade_3_5state.fasta")

lines, hosts = [], []
for l in src.read_text().splitlines():
    if l.startswith(">"):
        acc, host, year = l[1:].split()[0].split("|")
        if host == "ailurid":
            host = "wild_felid"      # single observation folded in
        hosts.append(host)
        lines.append(f">{acc}|{host}|{year}")
    elif l.strip():
        lines.append(l.strip())
out.write_text("\n".join(lines) + "\n")

print(f"{len(hosts)} sequences -> {out.name}")
print(dict(Counter(hosts)))
print("states:", len(set(hosts)), "-> rateIndicator dimension should be",
      len(set(hosts))*(len(set(hosts))-1)//2, "if symmetric")

In [ ]:
from pathlib import Path
d = Path("data/processed")
print("in data/processed:")
for f in sorted(d.glob("*.fasta")):
    print(f"  {f.name:<40} {f.stat().st_size/1024:>8.1f} KB")

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter

p = Path("beast/clade3/clade3_5state.xml")
raw = p.read_text(); r = ET.fromstring(raw)

print("partitions   :", len(r.findall(".//data")))
print("chainLength  :", [x.get("chainLength") for x in r.findall(".//run")])
print("rateIndicator:", raw.count("rateIndicator"), "(want 0)")
import re
m = re.search(r'rateIndicator[^>]*dimension="(\d+)"', raw)
print("  dimension  :", m.group(1) if m else "none — BSSVS is off")
print("loggers      :", [x.get("fileName") for x in r.findall(".//logger") if x.get("fileName")])

for el in r.iter():
    tn = el.get("traitname")
    if tn and not tn.startswith("date"):
        val = el.get("value") or (el.text or "")
        pairs = [y.strip() for y in val.replace("\n"," ").split(",") if "=" in y]
        print("trait states :", dict(Counter(y.rsplit('=',1)[1] for y in pairs)))

In [ ]:
from pathlib import Path
raw = Path("beast/clade3/clade3_5state.xml").read_text()
for kw in ["indicatorFlip", "BSSVSoperator", "BitFlip", "nonZeroRates"]:
    print(f"  {kw:<20} {raw.count(kw)}")
print("\nrateIndicator lines:")
for l in raw.splitlines():
    if "rateIndicator" in l:
        print("   ", l.strip()[:130])

In [ ]:
import shutil, subprocess, time, re
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

XML         = Path("beast/clade3/clade3_5state.xml").resolve()
SEEDS       = [12345, 54321]
THREADS     = 4
TIMEOUT_MIN = 150

# ---------- verify before committing ----------
raw = XML.read_text(); r = ET.fromstring(raw)
chain = re.search(r'chainLength="(\d+)"', raw)
total = int(chain.group(1)) if chain else None
states = {}
for el in r.iter():
    tn = el.get("traitname")
    if tn and not tn.startswith("date"):
        val = el.get("value") or (el.text or "")
        pairs = [y.strip() for y in val.replace("\n", " ").split(",") if "=" in y]
        if pairs:
            states = dict(Counter(y.rsplit('=', 1)[1] for y in pairs))

print(f"{XML.name}")
print(f"  partitions  : {len(r.findall('.//data'))}")
print(f"  chainLength : {total:,}" if total else "  chainLength : ?")
print(f"  trait states: {len(states)} -> {states}")
print(f"  BSSVS ops   : {sum(raw.count(k) for k in ('indicatorFlip','BSSVSoperator','BitFlip'))} (want 0)")
assert len(states) == 5, "expected 5 trait states"
assert sum(raw.count(k) for k in ("indicatorFlip", "BSSVSoperator", "BitFlip")) == 0, "BSSVS still active"

# ---------- park old outputs ----------
old = XML.parent / "previous_runs"
for f in XML.parent.glob("clade3_5state*"):
    if f.is_file() and f.suffix in (".log", ".trees"):
        old.mkdir(exist_ok=True); shutil.move(str(f), old / f.name)

def find_beast():
    exe = shutil.which("beast")
    if exe: return exe
    for rt in [Path("/Applications"), Path.home()/"Applications", Path.home()]:
        if rt.is_dir():
            for pat in ("BEAST*/bin/beast", "*/BEAST*/bin/beast"):
                hits = sorted(rt.glob(pat))
                if hits: return str(hits[0])
    return None

beast = find_beast()
if not beast:
    raise RuntimeError("BEAST not found")

# ---------- run both chains ----------
for seed in SEEDS:
    print("\n" + "="*66); print(f"  CHAIN seed={seed}"); print("="*66)
    t0, errors, posteriors, tail = time.time(), 0, [], []
    marks = {300_000: False, 1_500_000: False}
    proc = subprocess.Popen([beast, "-seed", str(seed), "-threads", str(THREADS),
                             "-overwrite", str(XML)],
                            cwd=str(XML.parent), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout:
            line = line.rstrip(); tail.append(line); tail[:] = tail[-30:]
            if "randomchoiceunnormalized" in line.lower():
                errors += 1
                if errors <= 2: print(f"  !! trait error {errors} — model still unstable")
            parts = line.split()
            if len(parts) >= 2 and parts[0].isdigit():
                state = int(parts[0])
                for mark in marks:
                    if state > mark and not marks[mark] and errors == 0:
                        marks[mark] = True
                        print(f"  >> past {mark:,} clean "
                              f"({'first crash point' if mark == 300_000 else 'second crash point'})")
                try:
                    posteriors.append(float(parts[1]))
                    if len(posteriors) % 50 == 1 and total:
                        el = time.time() - t0; frac = state/total
                        eta = (el/frac - el)/60 if frac > 0.01 else 0
                        print(f"  [{el/60:5.1f} min] {state:>12,}  {parts[1]:>12}  ETA {eta:5.1f} min")
                except ValueError:
                    pass
            if time.time() - t0 > TIMEOUT_MIN*60:
                proc.kill(); print("  KILLED on timeout"); break
    finally:
        proc.wait()

    print(f"  exit {proc.returncode} in {(time.time()-t0)/60:.1f} min | trait errors: {errors}")
    if proc.returncode != 0:
        print("  last lines:\n    " + "\n    ".join(tail[-8:])); break

    for f in ["clade3_5state.log", "clade3_5state.trees", "clade3_5state.host.trees"]:
        p = XML.parent / f
        if p.exists():
            p.rename(XML.parent / f.replace(".", f"_seed{seed}.", 1))

print("\nfinal outputs:")
for f in sorted(XML.parent.glob("*seed*")):
    print(f"  {f.name:<48} {f.stat().st_size/1024:>10.1f} KB")

In [ ]:
from pathlib import Path
d = Path("beast/clade3")
for f in sorted(d.glob("*seed12345*")):
    print(f"  {f.name:<48} {f.stat().st_size/1024:>10.1f} KB")

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import re

XML = Path("beast/clade3/clade3_5state.xml")
raw = XML.read_text()

print(f"{XML.name}\n")
print(f"UpDown occurrences : {raw.lower().count('updown')}")

# list every operator so we can see what's actually moving the tree and clock
print("\noperators:")
root = ET.fromstring(raw)
for op in root.findall(".//operator"):
    spec = op.get("spec", "")
    oid  = op.get("id", "")
    wt   = op.get("weight", "")
    targets = [op.get(k) for k in ("parameter", "tree", "up", "down") if op.get(k)]
    print(f"  {oid:<44} {spec:<28} w={wt:<6} {' '.join(targets)}")

# which operators touch the clock rate and tree height?
print("\noperators touching ucldMean:")
for op in

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import re

XML = Path("beast/clade3/clade3_5state.xml")
raw = XML.read_text()

print(f"{XML.name}\n")
print(f"UpDown occurrences : {raw.lower().count('updown')}")

# list every operator so we can see what's actually moving the tree and clock
print("\noperators:")
root = ET.fromstring(raw)
for op in root.findall(".//operator"):
    spec = op.get("spec", "")
    oid  = op.get("id", "")
    wt   = op.get("weight", "")
    targets = [op.get(k) for k in ("parameter", "tree", "up", "down") if op.get(k)]
    print(f"  {oid:<44} {spec:<28} w={wt:<6} {' '.join(targets)}")

# which operators touch the clock rate and tree height?
print("\noperators touching ucldMean:")
for op in root.findall(".//operator"):
    if "ucldMean" in ET.tostring(op, encoding="unicode"):
        print(f"  {op.get('id')}  ({op.get('spec')}, weight {op.get('weight')})")

print("\noperators touching the tree:")
for op in root.findall(".//operator"):
    if op.get("tree") or "Tree.t" in ET.tostring(op, encoding="unicode"):
        print(f"  {op.get('id')}  ({op.get('spec')}, weight {op.get('weight')})")

In [ ]:
import re, shutil, subprocess, os, stat, shlex
from pathlib import Path

SRC       = Path("beast/clade3/clade3_5state.xml").resolve()
CHAIN     = 100_000_000
LOG_EVERY = 100_000          # keeps 1000 samples per file at 100M
SEEDS     = [12345, 54321]
THREADS   = 4
RUNDIR    = SRC.parent / "run100M"

def find_beast():
    exe = shutil.which("beast")
    if exe: return exe
    for rt in [Path("/Applications"), Path.home()/"Applications", Path.home()]:
        if rt.is_dir():
            for pat in ("BEAST*/bin/beast", "*/BEAST*/bin/beast"):
                hits = sorted(rt.glob(pat))
                if hits: return str(hits[0])
    raise RuntimeError("BEAST not found")

beast = find_beast()

raw = SRC.read_text()
raw, n1 = re.subn(r'(chainLength=")\d+(")', rf'\g<1>{CHAIN}\g<2>', raw)
def fix(m):
    return m.group(0) if 'id="screenlog"' in m.group(0) else re.sub(
        r'logEvery="\d+"', f'logEvery="{LOG_EVERY}"', m.group(0))
raw, n2 = re.subn(r'<logger\b[^>]*>', fix, raw)
print(f"chainLength -> {CHAIN:,} ({n1} edit), logEvery -> {LOG_EVERY:,} ({n2} loggers)")

# each chain gets its own directory, so the fixed output filenames don't collide
for seed in SEEDS:
    sd = RUNDIR / f"seed{seed}"
    sd.mkdir(parents=True, exist_ok=True)
    (sd / SRC.name).write_text(raw)

lines = ["#!/bin/bash", "set -e"]
for seed in SEEDS:
    sd = RUNDIR / f"seed{seed}"
    lines += [f'echo "=== seed {seed} started $(date) ==="',
              f'cd {shlex.quote(str(sd))}',
              f'{shlex.quote(beast)} -seed {seed} -threads {THREADS} -overwrite '
              f'{shlex.quote(str(sd / SRC.name))} > run.out 2>&1',
              f'echo "=== seed {seed} finished $(date) ==="']
sh = RUNDIR / "run_all.sh"
sh.write_text("\n".join(lines) + "\n")
sh.chmod(sh.stat().st_mode | stat.S_IEXEC)

# caffeinate stops the Mac sleeping; nohup detaches from the kernel
proc = subprocess.Popen(
    f"nohup caffeinate -i {shlex.quote(str(sh))} > {shlex.quote(str(RUNDIR/'driver.log'))} 2>&1 &",
    shell=True, cwd=str(RUNDIR))

print(f"\nlaunched, pid group detached")
print(f"driver log : {RUNDIR/'driver.log'}")
print(f"chain logs : {RUNDIR}/seed*/run.out")
print(f"\nEstimated ~9 h per chain, ~18 h total. Safe to close this notebook.")
print("Leave the laptop plugged in. Closing the lid still sleeps it — use")
print("System Settings > Lock Screen, or just leave it open.")

In [ ]:
import os
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "scripts").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("cwd:", Path.cwd())

In [ ]:
import os, re, shutil, subprocess, stat, shlex, sys
from pathlib import Path

# ---------- 1. get to the repo root ----------
ROOT = Path.cwd()
while not (ROOT / "scripts").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "scripts").is_dir():
    raise RuntimeError(f"repo root not found from {Path.cwd()} — set it manually")
os.chdir(ROOT)
print(f"repo root : {ROOT}")

# ---------- 2. environment report ----------
print(f"python    : {sys.executable}")
in_env = "cdv-phylo" in sys.executable
print(f"kernel env: {'cdv-phylo' if in_env else 'NOT cdv-phylo (base?)'}")
if not in_env:
    print("  ^ fine for launching BEAST (it is a separate Java app).")
    print("    Switch kernels tomorrow before any pipeline work.")

# ---------- 3. find BEAST ----------
def find_beast():
    exe = shutil.which("beast")
    if exe: return exe
    for rt in [Path("/Applications"), Path.home()/"Applications", Path.home()]:
        if rt.is_dir():
            for pat in ("BEAST*/bin/beast", "*/BEAST*/bin/beast"):
                hits = sorted(rt.glob(pat))
                if hits: return str(hits[0])
    raise RuntimeError("BEAST not found — set `beast` manually")

beast = find_beast()
print(f"beast     : {beast}")

# ---------- 4. build the 100M XMLs ----------
SRC       = (ROOT / "beast/clade3/clade3_5state.xml").resolve()
CHAIN     = 100_000_000
LOG_EVERY = 100_000
SEEDS     = [12345, 54321]
THREADS   = 4
RUNDIR    = SRC.parent / "run100M"

if not SRC.is_file():
    raise FileNotFoundError(f"{SRC} missing. In {SRC.parent}: "
        f"{sorted(p.name for p in SRC.parent.iterdir()) if SRC.parent.is_dir() else 'dir missing'}")

raw = SRC.read_text()
raw, n_chain = re.subn(r'(chainLength=")\d+(")', rf'\g<1>{CHAIN}\g<2>', raw)
def fix(m):
    return m.group(0) if 'id="screenlog"' in m.group(0) else re.sub(
        r'logEvery="\d+"', f'logEvery="{LOG_EVERY}"', m.group(0))
raw, n_log = re.subn(r'<logger\b[^>]*>', fix, raw)

print(f"\nchainLength -> {CHAIN:,}   ({n_chain} edit, want 1)")
print(f"logEvery    -> {LOG_EVERY:,}   ({n_log} loggers scanned, want 4)")
assert n_chain == 1, "chainLength not edited — check the XML"

for seed in SEEDS:
    sd = RUNDIR / f"seed{seed}"
    sd.mkdir(parents=True, exist_ok=True)
    (sd / SRC.name).write_text(raw)
print(f"run dirs   : {[p.name for p in sorted(RUNDIR.glob('seed*'))]}")

# ---------- 5. driver script ----------
lines = ["#!/bin/bash", "set -e"]
for seed in SEEDS:
    sd = RUNDIR / f"seed{seed}"
    lines += [f'echo "=== seed {seed} started $(date) ==="',
              f'cd {shlex.quote(str(sd))}',
              f'{shlex.quote(beast)} -seed {seed} -threads {THREADS} -overwrite '
              f'{shlex.quote(str(sd / SRC.name))} > run.out 2>&1',
              f'echo "=== seed {seed} finished $(date) ==="']
sh = RUNDIR / "run_all.sh"
sh.write_text("\n".join(lines) + "\n")
sh.chmod(sh.stat().st_mode | stat.S_IEXEC)

# ---------- 6. launch, detached ----------
subprocess.Popen(
    f"nohup caffeinate -i {shlex.quote(str(sh))} > {shlex.quote(str(RUNDIR/'driver.log'))} 2>&1 &",
    shell=True, cwd=str(RUNDIR))

print(f"\nLAUNCHED — detached from this notebook")
print(f"  driver log : {RUNDIR/'driver.log'}")
print(f"  chain output: {RUNDIR}/seed*/run.out")
print(f"\n~9 h per chain, ~18 h total. Safe to close Jupyter.")
print("Keep it plugged in and leave the lid open — caffeinate stops idle sleep,")
print("not lid-close sleep.")

In [ ]:
from pathlib import Path
import time

RUNDIR = Path("$HOME/projects/cdv-phylodynamics/beast/clade3/run100M")
TOTAL = 100_000_000

drv = RUNDIR / "driver.log"
print(drv.read_text() if drv.exists() else "no driver log yet")
for sd in sorted(RUNDIR.glob("seed*")):
    log, out = sd / "clade3_5state.log", sd / "run.out"
    print(f"\n{sd.name}")
    if not out.exists():
        print("  not started yet"); continue
    if log.exists():
        rows = [l for l in log.read_text().splitlines()
                if l and not l.startswith("#") and l.split()[0].isdigit()]
        if rows:
            state = int(rows[-1].split()[0])
            print(f"  state {state:,} / {TOTAL:,}  ({state/TOTAL:.1%})")
            print(f"  last write {(time.time()-out.stat().st_mtime)/60:.0f} min ago")
        else:
            print("  log exists, no samples yet (normal in the first few minutes)")
    tail = out.read_text().splitlines()[-3:]
    print("  " + "\n  ".join(t[:90] for t in tail))

In [ ]:
from pathlib import Path
import statistics, time

RUNDIR = Path("$HOME/projects/cdv-phylodynamics/beast/clade3/run100M")
TOTAL  = 100_000_000

def ess_rough(x):
    """Approximate ESS via autocorrelation. Indicative only — use Tracer for the real number."""
    n = len(x)
    if n < 10: return 0
    m, v = statistics.mean(x), statistics.pvariance(x)
    if v == 0: return 0
    s = 0.0
    for lag in range(1, min(n//3, 500)):
        c = sum((x[i]-m)*(x[i+lag]-m) for i in range(n-lag))/(n-lag)/v
        if c < 0.05: break
        s += c
    return n/(1+2*s)

drv = RUNDIR / "driver.log"
print(drv.read_text() if drv.exists() else "no driver log\n")

for sd in sorted(RUNDIR.glob("seed*")):
    print(f"--- {sd.name} ---")
    log, out = sd / "clade3_5state.log", sd / "run.out"
    if not log.is_file():
        print("  no log yet\n"); continue
    lines = [l for l in log.read_text().splitlines() if l and not l.startswith("#")]
    hdr = lines[0].split("\t")
    rows = [l.split("\t") for l in lines[1:] if l.split("\t")[0].isdigit()]
    if not rows:
        print("  no samples yet\n"); continue

    state = int(rows[-1][0])
    done = state >= TOTAL*0.999
    age = (time.time() - out.stat().st_mtime)/60 if out.exists() else None
    print(f"  {len(rows)} samples, last state {state:,} ({state/TOTAL:.1%})")
    print(f"  {'COMPLETE' if done else 'still running'}"
          + (f" — last write {age:.0f} min ago" if age is not None else ""))
    if not done and age is not None and age > 20:
        print("  !! no output for 20+ min — check it hasn't died")

    burn = len(rows)//10
    print(f"  (post 10% burn-in, {len(rows)-burn} samples)")
    for col in hdr:
        if any(k in col for k in ("posterior", "Tree.height", "ucldMean",
                                  "traitClockRate", "ucldStdev", "popSize")):
            i = hdr.index(col)
            try:
                vals = [float(r[i]) for r in rows[burn:]]
            except (ValueError, IndexError):
                continue
            e = ess_rough(vals)
            flag = "" if e >= 200 else "   <-- LOW"
            print(f"  {col:<34} mean {statistics.mean(vals):>12.5g}  ESS~{e:>6.0f}{flag}")

    for f in sorted(sd.glob("*")):
        if f.suffix in (".log", ".trees"):
            print(f"    {f.name:<36} {f.stat().st_size/1024:>9.1f} KB")
    print()

In [ ]:
from pathlib import Path
import time

sd = Path("$HOME/projects/cdv-phylodynamics/beast/clade3/run100M/seed54321")
rows = [l for l in (sd/"clade3_5state.log").read_text().splitlines()
        if l and not l.startswith("#") and l.split()[0].isdigit()]
state = int(rows[-1].split()[0])
mins_per_M = 5.87
remaining = (100_000_000 - state) / 1_000_000 * mins_per_M
print(f"state {state:,} ({state/1e8:.1%})")
print(f"~{remaining/60:.1f} hours remaining")
print(f"finishing around {time.strftime('%H:%M', time.localtime(time.time()+remaining*60))}")

In [ ]:
import shutil, subprocess, time
from pathlib import Path

RUNDIR = Path("$HOME/projects/cdv-phylodynamics/beast/clade3/run100M")
SEEDS  = [12345, 54321]
BURNIN = 10          # percent, applied per chain by LogCombiner

def find_tool(name):
    exe = shutil.which(name)
    if exe: return exe
    for rt in [Path("/Applications"), Path.home()/"Applications", Path.home()]:
        if rt.is_dir():
            for pat in (f"BEAST*/bin/{name}", f"*/BEAST*/bin/{name}"):
                hits = sorted(rt.glob(pat))
                if hits: return str(hits[0])
    raise RuntimeError(f"{name} not found — it ships in BEAST*/bin/")

logcombiner   = find_tool("logcombiner")
treeannotator = find_tool("treeannotator")
print(f"logcombiner   : {logcombiner}")
print(f"treeannotator : {treeannotator}\n")

# --- confirm both chains completed ---
for seed in SEEDS:
    log = RUNDIR / f"seed{seed}" / "clade3_5state.log"
    rows = [l for l in log.read_text().splitlines()
            if l and not l.startswith("#") and l.split()[0].isdigit()]
    state = int(rows[-1].split()[0])
    print(f"seed{seed}: {len(rows)} samples, final state {state:,}"
          + ("  COMPLETE" if state >= 99_000_000 else "  INCOMPLETE"))
    assert state >= 99_000_000, f"seed{seed} did not finish"

def run(cmd, label):
    print(f"\n=== {label} ===")
    print("$ " + " ".join(str(c) for c in cmd))
    t0 = time.time()
    p = subprocess.run([str(c) for c in cmd], cwd=str(RUNDIR),
                       capture_output=True, text=True)
    out = (p.stdout or "") + (p.stderr or "")
    print(out[-1200:] if p.returncode else out[-400:])
    print(f"exit {p.returncode} in {(time.time()-t0)/60:.1f} min")
    if p.returncode != 0:
        raise RuntimeError(f"{label} failed")

# --- combine parameter logs ---
run([logcombiner,
     "-log", f"seed{SEEDS[0]}/clade3_5state.log",
     "-log", f"seed{SEEDS[1]}/clade3_5state.log",
     "-b", BURNIN, "-o", "combined.log"], "LogCombiner: parameter logs")

# --- combine host trees (the ancestral state reconstruction) ---
run([logcombiner,
     "-log", f"seed{SEEDS[0]}/clade3_5state.host.trees",
     "-log", f"seed{SEEDS[1]}/clade3_5state.host.trees",
     "-b", BURNIN, "-o", "combined.host.trees"], "LogCombiner: host trees")

# --- MCC tree. burnin 0: LogCombiner already stripped it ---
run([treeannotator, "-burnin", 0, "-heights", "median",
     "combined.host.trees", "clade3_mcc.tree"], "TreeAnnotator: MCC tree")

print("\noutputs:")
for f in ["combined.log", "combined.host.trees", "clade3_mcc.tree"]:
    p = RUNDIR / f
    print(f"  {f:<26} {p.stat().st_size/1024:>10.1f} KB" if p.exists()
          else f"  {f:<26}  MISSING")

rows = [l for l in (RUNDIR/"combined.log").read_text().splitlines()
        if l and not l.startswith("#") and l.split()[0].isdigit()]
print(f"\ncombined trace: {len(rows)} samples (expect ~1800)")
print(f"\nNext: open {RUNDIR/'combined.log'} in Tracer for the final HPDs,")
print(f"and {RUNDIR/'clade3_mcc.tree'} in FigTree, coloured by host state.")

In [ ]:
from pathlib import Path
RUNDIR = Path("$HOME/projects/cdv-phylodynamics/beast/clade3/run100M")
for f in sorted(RUNDIR.iterdir()):
    if f.is_file():
        print(f"  {f.name:<34} {f.stat().st_size/1024:>10.1f} KB")

In [ ]:
import re
from pathlib import Path
from collections import Counter

MCC = Path("$HOME/projects/cdv-phylodynamics/beast/clade3/run100M/clade3_mcc.tree")
raw = MCC.read_text()

# annotations look like [&location="procyonid",location.prob=0.83,height=41.2,posterior=0.97,...]
blocks = re.findall(r'\[&([^\]]*)\]', raw)
print(f"{len(blocks)} annotated nodes\n")

nodes = []
for b in blocks:
    d = {}
    for m in re.finditer(r'(\w[\w.]*)=("?)([^",]+)\2', b):
        d[m.group(1)] = m.group(3)
    if "location" in d:
        nodes.append(d)

print(f"{len(nodes)} with a location annotation\n")

print("STATE COUNTS ACROSS ALL NODES")
for s, n in Counter(x["location"] for x in nodes).most_common():
    print(f"  {s:<16} {n:>4}")

print("\nCONFIDENCE DISTRIBUTION (location.prob)")
probs = [float(x["location.prob"]) for x in nodes if "location.prob" in x]
for lo, hi in [(0.0,0.4),(0.4,0.6),(0.6,0.8),(0.8,0.95),(0.95,1.01)]:
    n = sum(1 for p in probs if lo <= p < hi)
    print(f"  {lo:.2f}-{hi:.2f}   {n:>4}  ({n/len(probs):.0%})")
print(f"\n  median prob: {sorted(probs)[len(probs)//2]:.3f}")
print(f"  above 0.80 : {sum(1 for p in probs if p >= 0.8)} ({sum(1 for p in probs if p>=0.8)/len(probs):.0%})")

print("\nDEEPEST 25 NODES (oldest first) — the backbone")
deep = sorted([x for x in nodes if "height" in x],
              key=lambda x: -float(x["height"]))[:25]
print(f"  {'height':>8} {'year':>7}  {'state':<16} {'prob':>6}  {'posterior':>9}")
for x in deep:
    h = float(x["height"])
    print(f"  {h:>8.1f} {2023.6-h:>7.0f}  {x['location']:<16} "
          f"{float(x.get('location.prob',0)):>6.3f}  {x.get('posterior','-'):>9}")

print("\nWELL-SUPPORTED DEEP NODES (height>40, prob>=0.8)")
good = [x for x in nodes if float(x.get("height",0)) > 40
        and float(x.get("location.prob",0)) >= 0.8]
print(f"  {len(good)} nodes")
for s, n in Counter(x["location"] for x in good).most_common():
    print(f"    {s:<16} {n:>3}")

In [ ]:
import re, xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

XML = Path("beast/clade3/clade3_5state_fixed.xml").resolve()
raw = XML.read_text(); root = ET.fromstring(raw)

print(f"{XML.name}")
print(f"  partitions   : {len(root.findall('.//data'))}  (want 2)")
print(f"  chainLength  : {[x.get('chainLength') for x in root.findall('.//run')]}")
print(f"  BSSVS ops    : {sum(raw.count(k) for k in ('indicatorFlip','BSSVSoperator','BitFlip'))}  (want 0)")

# --- the critical check: date direction ---
dates = {}
states = {}
for el in root.iter():
    tn = el.get("traitname")
    if not tn: continue
    val = el.get("value") or (el.text or "")
    pairs = [y.strip() for y in val.replace("\n", " ").split(",") if "=" in y]
    if not pairs: continue
    if tn.startswith("date"):
        print(f"\n  traitname    : '{tn}'")
        for y in pairs:
            taxon, v = y.rsplit("=", 1)
            try: dates[taxon.strip()] = float(v)
            except ValueError: pass
    else:
        states = Counter(y.rsplit("=", 1)[1] for y in pairs)

print(f"  trait states : {len(states)} -> {dict(states)}")

if dates:
    newest = max(dates.values()); oldest = min(dates.values())
    print(f"\n  date range   : {oldest:.3f} to {newest:.3f}")
    print(f"  span         : {newest-oldest:.1f} years")
    print("\n  DIRECTION CHECK")
    print(f"    newest sequence should be {newest:.1f}, oldest {oldest:.1f}")
    print("    BEAST must treat the LARGER number as MORE RECENT.")
    tn_ok = "forward" in [e.get("traitname") for e in root.iter() if e.get("traitname")][0].lower() \
            if any(e.get("traitname") for e in root.iter()) else False
    tnames = [e.get("traitname") for e in root.iter() if e.get("traitname")]
    dtn = [t for t in tnames if t and t.startswith("date")]
    print(f"    traitname is '{dtn[0] if dtn else '?'}'")
    if dtn and dtn[0] == "date-forward":
        print("    OK — 'date-forward' means larger = more recent")
    elif dtn and dtn[0] == "date-backward":
        print("    WRONG — 'date-backward' means larger = older. Fix in BEAUti.")
    else:
        print("    'date' is ambiguous. Confirm in BEAUti that the 2023 sequence")
        print("    shows height 0 in the date table before running.")

In [ ]:
import os
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "scripts").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("cwd:", Path.cwd(), "\n")

d = ROOT / "beast/clade3"
for f in sorted(d.glob("*.xml")):
    print(f"  {f.name:<40} {f.stat().st_size/1024:>9.1f} KB")

In [ ]:
from pathlib import Path
src = ROOT / "beast/clade3/clade3_5state.xml"
raw = src.read_text()
print("date traitname occurrences:", raw.count('traitname="date"'))
i = raw.find('traitname="date"')
print(raw[max(0,i-200):i+300])

In [ ]:
import re
from pathlib import Path

src = ROOT / "beast/clade3/clade3_5state.xml"
raw = src.read_text()

print("all traitname attributes found:")
for m in re.finditer(r'traitname\s*=\s*["\']([^"\']+)["\']', raw):
    print(f"  {m.group(1)!r}  at char {m.start()}")

print("\ntraitSet / trait elements in full:")
for m in re.finditer(r'<(traitSet|trait)\b[^>]*>', raw):
    print(f"\n  {m.group(0)[:300]}")

print("\nlines mentioning 'date':")
for l in raw.splitlines():
    if re.search(r'\bdate\b', l, re.I) and 'traitname' in l.lower() or 'dateTrait' in l:
        print("  ", l.strip()[:200])

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET, re, shutil

src = ROOT / "beast/clade3/clade3_5state.xml"
dst = ROOT / "beast/clade3/clade3_5state_fixed.xml"

raw = src.read_text()
n = raw.count('traitname="date-backward"')
assert n == 1, f"expected 1 occurrence, found {n}"
raw = raw.replace('traitname="date-backward"', 'traitname="date-forward"')
ET.fromstring(raw)                       # fails loudly if the edit broke the XML
dst.write_text(raw)

# verify
check = dst.read_text()
print(f"wrote {dst.name}")
print(f"  date-backward remaining: {check.count('date-backward')}  (want 0)")
print(f"  date-forward           : {check.count('date-forward')}  (want 1)")

# confirm the date values themselves are untouched
m = re.search(r'traitname="date-forward" value="([^"]{0,200})', check)
print(f"\n  first dates: {m.group(1)[:120]}...")
vals = re.findall(r'=(\d{4}\.\d+)', check[m.start():m.start()+40000])
vals = [float(v) for v in vals]
print(f"  {len(vals)} dates, range {min(vals):.3f} - {max(vals):.3f}")
print(f"\n  With date-forward, BEAST will treat {max(vals):.1f} as the most recent")
print(f"  and {min(vals):.1f} as the oldest — which is correct.")

In [ ]:
SRC    = (ROOT / "beast/clade3/clade3_5state_fixed.xml").resolve()
RUNDIR = SRC.parent / "run100M_fixed"

In [ ]:
import os, re, shutil, subprocess, stat, shlex, sys
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

# ---------- repo root ----------
ROOT = Path.cwd()
while not (ROOT / "scripts").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "scripts").is_dir():
    raise RuntimeError(f"repo root not found from {Path.cwd()}")
os.chdir(ROOT)
print(f"repo root : {ROOT}")

SRC       = (ROOT / "beast/clade3/clade3_5state_fixed.xml").resolve()
RUNDIR    = SRC.parent / "run100M_fixed"
CHAIN     = 100_000_000
LOG_EVERY = 100_000
SEEDS     = [12345, 54321]
THREADS   = 4

if not SRC.is_file():
    raise FileNotFoundError(f"{SRC.name} missing — run the date-direction fix cell first")

# ---------- pre-flight: refuse to launch a broken analysis ----------
raw = SRC.read_text()
root = ET.fromstring(raw)

dtn = [e.get("traitname") for e in root.iter()
       if e.get("traitname") and e.get("traitname").startswith("date")]
states = {}
for el in root.iter():
    tn = el.get("traitname")
    if tn and not tn.startswith("date"):
        val = el.get("value") or (el.text or "")
        pairs = [y.strip() for y in val.replace("\n"," ").split(",") if "=" in y]
        if pairs:
            states = Counter(y.rsplit("=",1)[1] for y in pairs)

print(f"\nPRE-FLIGHT")
print(f"  date traitname : {dtn[0] if dtn else 'NONE'}")
print(f"  partitions     : {len(root.findall('.//data'))}")
print(f"  trait states   : {len(states)} -> {dict(states)}")
print(f"  BSSVS ops      : {sum(raw.count(k) for k in ('indicatorFlip','BSSVSoperator','BitFlip'))}")

assert dtn and dtn[0] == "date-forward", f"date direction is {dtn}, must be date-forward"
assert len(states) == 5, f"expected 5 trait states, got {len(states)}"
assert sum(raw.count(k) for k in ("indicatorFlip","BSSVSoperator","BitFlip")) == 0, "BSSVS still on"
print("  all checks passed")

# ---------- set chain length ----------
raw, n1 = re.subn(r'(chainLength=")\d+(")', rf'\g<1>{CHAIN}\g<2>', raw)
def fix(m):
    return m.group(0) if 'id="screenlog"' in m.group(0) else re.sub(
        r'logEvery="\d+"', f'logEvery="{LOG_EVERY}"', m.group(0))
raw, n2 = re.subn(r'<logger\b[^>]*>', fix, raw)
print(f"\n  chainLength -> {CHAIN:,} ({n1} edit)")
print(f"  logEvery    -> {LOG_EVERY:,} ({n2} loggers)")
assert n1 == 1

for seed in SEEDS:
    sd = RUNDIR / f"seed{seed}"
    sd.mkdir(parents=True, exist_ok=True)
    (sd / SRC.name).write_text(raw)

# ---------- launch ----------
def find_beast():
    exe = shutil.which("beast")
    if exe: return exe
    for rt in [Path("/Applications"), Path.home()/"Applications", Path.home()]:
        if rt.is_dir():
            for pat in ("BEAST*/bin/beast", "*/BEAST*/bin/beast"):
                hits = sorted(rt.glob(pat))
                if hits: return str(hits[0])
    raise RuntimeError("BEAST not found")

beast = find_beast()
lines = ["#!/bin/bash", "set -e"]
for seed in SEEDS:
    sd = RUNDIR / f"seed{seed}"
    lines += [f'echo "=== seed {seed} started $(date) ==="',
              f'cd {shlex.quote(str(sd))}',
              f'{shlex.quote(beast)} -seed {seed} -threads {THREADS} -overwrite '
              f'{shlex.quote(str(sd / SRC.name))} > run.out 2>&1',
              f'echo "=== seed {seed} finished $(date) ==="']
sh = RUNDIR / "run_all.sh"
sh.write_text("\n".join(lines) + "\n")
sh.chmod(sh.stat().st_mode | stat.S_IEXEC)

subprocess.Popen(
    f"nohup caffeinate -i {shlex.quote(str(sh))} > {shlex.quote(str(RUNDIR/'driver.log'))} 2>&1 &",
    shell=True, cwd=str(RUNDIR))

print(f"\nLAUNCHED -> {RUNDIR}")
print(f"  beast: {beast}")
print(f"  ~10 h per chain. Safe to close Jupyter.")
print(f"\nCheck in ~10 minutes with the Tree.height sanity cell before leaving it.")

In [ ]:
#### TIME CHECK######

from pathlib import Path
sd = RUNDIR / "seed12345"
rows = [l for l in (sd/"clade3_5state.log").read_text().splitlines()
        if l and not l.startswith("#")]
hdr = rows[0].split("\t")
data = [r.split("\t") for r in rows[1:] if r.split("\t")[0].isdigit()]
i = hdr.index("Tree.height")
h = [float(r[i]) for r in data]
print(f"{len(h)} samples, Tree.height {min(h):.1f} - {max(h):.1f}")
print("sampling span is 31.6 years; tree height should exceed that")
print("but not wildly — a few hundred means something is still wrong")

In [ ]:
#### TIME CHECK######

from pathlib import Path
sd = RUNDIR / "seed12345"
rows = [l for l in (sd/"clade3_5state.log").read_text().splitlines()
        if l and not l.startswith("#")]
hdr = rows[0].split("\t")
data = [r.split("\t") for r in rows[1:] if r.split("\t")[0].isdigit()]
i = hdr.index("Tree.height")
h = [float(r[i]) for r in data]
print(f"{len(h)} samples, Tree.height {min(h):.1f} - {max(h):.1f}")
print("sampling span is 31.6 years; tree height should exceed that")
print("but not wildly — a few hundred means something is still wrong")

In [ ]:
import os, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "scripts").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)

RUNDIR = ROOT / "beast/clade3/run100M_fixed"
TOTAL  = 100_000_000
SPAN   = 31.6          # sampling span, for the sanity check

drv = RUNDIR / "driver.log"
print(drv.read_text().strip() if drv.exists() else "no driver log yet")

for sd in sorted(RUNDIR.glob("seed*")):
    print(f"\n{'='*54}\n{sd.name}\n{'='*54}")
    log, out = sd / "clade3_5state.log", sd / "run.out"
    if not out.exists():
        print("  not started"); continue
    if not log.exists():
        print("  starting up, no log yet"); continue

    rows = [l for l in log.read_text().splitlines() if l and not l.startswith("#")]
    hdr  = rows[0].split("\t")
    data = [r.split("\t") for r in rows[1:] if r.split("\t")[0].isdigit()]
    if not data:
        print("  no samples yet"); continue

    state = int(data[-1][0])
    done  = state >= TOTAL * 0.999
    age   = (time.time() - out.stat().st_mtime) / 60

    print(f"  state {state:,} / {TOTAL:,}  ({state/TOTAL:.1%})   {len(data)} samples")
    print(f"  {'COMPLETE' if done else 'running'} — last write {age:.0f} min ago")
    if not done:
        elapsed = (time.time() - (out.stat().st_ctime)) / 60
        if state > 0 and elapsed > 1:
            rate = state / elapsed
            print(f"  rate ~{rate/1e6:.2f} M/min, ETA {(TOTAL-state)/rate/60:.1f} h")
    if not done and age > 25:
        print("  !! no output for 25+ min — check it hasn't stalled")

    # sanity: tree height must be plausible against the sampling span
    if "Tree.height" in hdr:
        i = hdr.index("Tree.height")
        h = [float(r[i]) for r in data]
        recent = h[len(h)//2:]
        print(f"\n  Tree.height  range {min(h):.1f} - {max(h):.1f}, "
              f"recent mean {sum(recent)/len(recent):.1f}")
        mean_h = sum(recent)/len(recent)
        if mean_h < SPAN:
            print("  !! shorter than the sampling span — impossible, check dates")
        elif mean_h > SPAN * 4:
            print("  !! much larger than expected — check the date direction")
        else:
            print("  OK — plausible for a 31.6-year sampling span")

    if "ucldMean.H_clade_3_5state" in hdr:
        i = hdr.index("ucldMean.H_clade_3_5state")
        v = [float(r[i]) for r in data][len(data)//2:]
        print(f"  ucldMean     recent mean {sum(v)/len(v):.3e}  "
              f"(TempEst gave 7.46e-04)")

    for f in sorted(sd.glob("*")):
        if f.suffix in (".log", ".trees"):
            print(f"    {f.name:<38} {f.stat().st_size/1024:>9.1f} KB")

In [ ]:
import os, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "scripts").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)

RUNDIR = ROOT / "beast/clade3/run100M_fixed"
TOTAL  = 100_000_000
SPAN   = 31.6          # sampling span, for the sanity check

drv = RUNDIR / "driver.log"
print(drv.read_text().strip() if drv.exists() else "no driver log yet")

for sd in sorted(RUNDIR.glob("seed*")):
    print(f"\n{'='*54}\n{sd.name}\n{'='*54}")
    log, out = sd / "clade3_5state.log", sd / "run.out"
    if not out.exists():
        print("  not started"); continue
    if not log.exists():
        print("  starting up, no log yet"); continue

    rows = [l for l in log.read_text().splitlines() if l and not l.startswith("#")]
    hdr  = rows[0].split("\t")
    data = [r.split("\t") for r in rows[1:] if r.split("\t")[0].isdigit()]
    if not data:
        print("  no samples yet"); continue

    state = int(data[-1][0])
    done  = state >= TOTAL * 0.999
    age   = (time.time() - out.stat().st_mtime) / 60

    print(f"  state {state:,} / {TOTAL:,}  ({state/TOTAL:.1%})   {len(data)} samples")
    print(f"  {'COMPLETE' if done else 'running'} — last write {age:.0f} min ago")
    if not done:
        elapsed = (time.time() - (out.stat().st_ctime)) / 60
        if state > 0 and elapsed > 1:
            rate = state / elapsed
            print(f"  rate ~{rate/1e6:.2f} M/min, ETA {(TOTAL-state)/rate/60:.1f} h")
    if not done and age > 25:
        print("  !! no output for 25+ min — check it hasn't stalled")

    # sanity: tree height must be plausible against the sampling span
    if "Tree.height" in hdr:
        i = hdr.index("Tree.height")
        h = [float(r[i]) for r in data]
        recent = h[len(h)//2:]
        print(f"\n  Tree.height  range {min(h):.1f} - {max(h):.1f}, "
              f"recent mean {sum(recent)/len(recent):.1f}")
        mean_h = sum(recent)/len(recent)
        if mean_h < SPAN:
            print("  !! shorter than the sampling span — impossible, check dates")
        elif mean_h > SPAN * 4:
            print("  !! much larger than expected — check the date direction")
        else:
            print("  OK — plausible for a 31.6-year sampling span")

    if "ucldMean.H_clade_3_5state" in hdr:
        i = hdr.index("ucldMean.H_clade_3_5state")
        v = [float(r[i]) for r in data][len(data)//2:]
        print(f"  ucldMean     recent mean {sum(v)/len(v):.3e}  "
              f"(TempEst gave 7.46e-04)")

    for f in sorted(sd.glob("*")):
        if f.suffix in (".log", ".trees"):
            print(f"    {f.name:<38} {f.stat().st_size/1024:>9.1f} KB")

In [ ]:
from pathlib import Path
sd = ROOT / "beast/clade3/run100M_fixed/seed12345"
rows = [l for l in (sd/"clade3_5state.log").read_text().splitlines() if l and not l.startswith("#")]
hdr = rows[0].split("\t"); data = [r.split("\t") for r in rows[1:] if r.split("\t")[0].isdigit()]
i = hdr.index("Tree.height")
h = [float(r[i]) for r in data[len(data)//10:]]
print(f"final state: {int(data[-1][0]):,}  |  {len(data)} samples")
print(f"Tree.height post-burnin: mean {sum(h)/len(h):.1f}, range {min(h):.1f}-{max(h):.1f}")
print(f"implied TMRCA: {2023.622 - sum(h)/len(h):.0f}")
print(f"sampling span 31.6 yr — heights must exceed that: {min(h) > 31.6}")

In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "scripts").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("repo root:", ROOT)

EMAIL = "<put your NCBI-registered email here>"
QUERY = ROOT / "data/processed/H_clade_3.fasta"      # adjust if yours is named differently

if not QUERY.is_file():
    print(f"\n{QUERY.name} not found. FASTA files available:")
    for f in sorted((ROOT/"data/processed").glob("*.fasta")):
        print(f"  {f.name}")
    raise FileNotFoundError("set QUERY to the right clade_3 FASTA")

if not (ROOT/"scripts/08_add_references.py").is_file():
    raise FileNotFoundError("08_add_references.py missing — download it from the zip into scripts/")

def run(cmd, cwd=ROOT):
    print("$", " ".join(str(c) for c in cmd), "\n")
    p = subprocess.run([str(c) for c in cmd], cwd=str(cwd),
                       capture_output=True, text=True)
    print(p.stdout or "")
    if p.returncode != 0:
        print("STDERR:\n", p.stderr[-2000:])
    return p.returncode

# ---- 1. fetch references and merge ----
rc = run([sys.executable, "scripts/08_add_references.py",
          "--email", EMAIL,
          "--query-fasta", str(QUERY.relative_to(ROOT))])
if rc != 0:
    raise RuntimeError("reference fetch failed — see output above")

combined = ROOT / "data/processed/clade3_with_refs.fasta"
aln      = ROOT / "data/processed/clade3_with_refs_aln.fasta"

# ---- 2. realign ----
mafft = shutil.which("mafft")
if not mafft:
    print("\nmafft not on PATH — run this in a terminal with the conda env active:")
    print(f"  mafft --auto {combined} > {aln}")
else:
    print("\naligning with MAFFT (a few minutes)...")
    with aln.open("w") as fh:
        p = subprocess.run([mafft, "--auto", "--adjustdirection", str(combined)],
                           stdout=fh, stderr=subprocess.PIPE, text=True)
    print("aligned" if p.returncode == 0 else p.stderr[-1500:])
    # strip MAFFT's _R_ prefix on reverse-complemented sequences
    txt = aln.read_text()
    if "_R_" in txt:
        n = txt.count(">_R_")
        aln.write_text(txt.replace(">_R_", ">"))
        print(f"  note: {n} sequences were reverse-complemented by MAFFT")

# ---- 3. tree ----
iqtree = shutil.which("iqtree2") or shutil.which("iqtree")
if not iqtree:
    print("\niqtree not on PATH — run in a terminal:")
    print(f"  iqtree2 -s {aln} -m MFP -B 1000 --alrt 1000 -T AUTO "
          f"--prefix data/processed/clade3_with_refs -redo")
else:
    print("\nbuilding ML tree (20-60 min)...")
    run([iqtree, "-s", str(aln), "-m", "MFP", "-B", "1000", "--alrt", "1000",
         "-T", "AUTO", "--prefix", "data/processed/clade3_with_refs", "-redo"])

print("\nWhen it finishes, open data/processed/clade3_with_refs.treefile and check:")
print("  1. Which REF_ tips fall INSIDE your clade -> your lineage name")
print("  2. Whether any of YOUR sequences fall in the America-1/vaccine clade")
print("  3. Whether the 4 Denmark sequences group with REF_Europe-wildlife_Danish-mink")

In [ ]:
grep -c "^>" data/processed/clade3_with_refs_aln.fasta
grep "^>REF_" data/processed/clade3_with_refs_aln.fasta | head -30

In [ ]:
from pathlib import Path
aln = ROOT / "data/processed/clade3_with_refs_aln.fasta"
heads = [l for l in aln.read_text().splitlines() if l.startswith(">")]
refs  = [h for h in heads if h.startswith(">REF_")]
print(f"{len(heads)} sequences, {len(refs)} references\n")
for r in refs:
    print(" ", r[1:])

In [ ]:
from pathlib import Path
aln = ROOT / "data/processed/clade3_with_refs_aln.fasta"

seqs, name, chunks = {}, None, []
for l in aln.read_text().splitlines():
    if l.startswith(">"):
        if name: seqs[name] = "".join(chunks)
        name, chunks = l[1:].split()[0], []
    elif l.strip(): chunks.append(l.strip())
seqs[name] = "".join(chunks)

dupes = ["REF_America-2_16407_EU716337|reference|2004",
         "REF_America-2_171391-513_KJ123771|reference|2004"]
removed = [d for d in dupes if d in seqs]
for d in removed: del seqs[d]

out = ROOT / "data/processed/clade3_with_refs_dedup.fasta"
with out.open("w") as fh:
    for k, v in seqs.items():
        fh.write(f">{k}\n")
        for i in range(0, len(v), 60): fh.write(v[i:i+60] + "\n")
print(f"removed {len(removed)} duplicate references")
print(f"{len(seqs)} sequences -> {out.name}")

In [ ]:
import os, subprocess, shutil, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "scripts").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)

ALN = ROOT / "data/processed/clade3_with_refs_dedup.fasta"
if not ALN.is_file():
    raise FileNotFoundError(f"{ALN.name} missing — run the dedup cell first")

heads = [l for l in ALN.read_text().splitlines() if l.startswith(">")]
print(f"{ALN.name}: {len(heads)} sequences, "
      f"{sum(1 for h in heads if h.startswith('>REF_'))} references")

iqtree = shutil.which("iqtree2") or shutil.which("iqtree")
if not iqtree:
    raise RuntimeError("iqtree not found — conda activate cdv-phylo")

cmd = [iqtree, "-s", "data/processed/clade3_with_refs_dedup.fasta",
       "-m", "MFP", "-B", "1000", "--alrt", "1000",
       "-T", "2",                       # leaves cores for the BEAST chain
       "--prefix", "data/processed/clade3_with_refs", "-redo"]

print("\n$ " + " ".join(cmd) + "\n")
t0 = time.time()
p = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
out = p.stdout + p.stderr
print(out[-2500:])
print(f"\nexit {p.returncode} in {(time.time()-t0)/60:.1f} min")

tf = ROOT / "data/processed/clade3_with_refs.treefile"
if tf.is_file():
    print(f"\ntree written: {tf}")
    import re
    iq = (ROOT / "data/processed/clade3_with_refs.iqtree").read_text()
    m = re.search(r"Best-fit model according to \w+: (\S+)", iq)
    print(f"model selected: {m.group(1) if m else '?'}")

In [ ]:
import os, shutil, subprocess, time, re
from pathlib import Path

ENV = "cdv-phylo"

ROOT = Path.cwd()
while not (ROOT / "scripts").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("repo root:", ROOT)

# ---------- find binaries in the conda env, even from a base kernel ----------
def find_tool(tool, env=ENV):
    p = shutil.which(tool)
    if p: return p
    roots = []
    if os.environ.get("CONDA_PREFIX"):
        roots.append(Path(os.environ["CONDA_PREFIX"]).parent.parent)
    roots += [Path.home()/"anaconda3", Path.home()/"miniforge3", Path.home()/"miniconda3",
              Path.home()/"opt/anaconda3"]
    for r in roots:
        if r.is_dir():
            hits = sorted(r.glob(f"envs/{env}/bin/{tool}"))
            if hits: return str(hits[0])
    return None

iqtree = find_tool("iqtree2") or find_tool("iqtree")
envpy  = find_tool("python")
print(f"iqtree : {iqtree or 'NOT FOUND'}")
if not iqtree:
    raise RuntimeError(f"iqtree not found in PATH or in the {ENV} env — "
                       "run it from a terminal with the env active instead")

# ---------- check the alignment ----------
ALN = ROOT / "data/processed/clade3_with_refs_dedup.fasta"
heads = [l for l in ALN.read_text().splitlines() if l.startswith(">")]
nref = sum(1 for h in heads if h.startswith(">REF_"))
print(f"\n{ALN.name}: {len(heads)} sequences ({nref} references, {len(heads)-nref} query)")
assert nref > 5, "too few references — the lineage question needs them"

# ---------- run ----------
cmd = [iqtree, "-s", "data/processed/clade3_with_refs_dedup.fasta",
       "-m", "MFP", "-B", "1000", "--alrt", "1000", "-T", "2",
       "--prefix", "data/processed/clade3_with_refs", "-redo"]
print("\n$ " + " ".join(cmd) + "\n")
t0 = time.time()
p = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
print((p.stdout + p.stderr)[-2500:])
print(f"\nexit {p.returncode} after {(time.time()-t0)/60:.1f} min")

tf = ROOT / "data/processed/clade3_with_refs.treefile"
if tf.is_file():
    iq = (ROOT / "data/processed/clade3_with_refs.iqtree").read_text()
    m = re.search(r"Best-fit model according to \w+: (\S+)", iq)
    print(f"\ntree  : {tf}")
    print(f"model : {m.group(1) if m else '?'}")
    print("\nOpen it in FigTree, or upload the .treefile and I'll parse the clade memberships.")

# ---------- optional: make the env available as a Jupyter kernel ----------
print("\n" + "-"*60)
if envpy:
    print("To stop hitting this problem, register the env as a kernel:")
    print(f'  {envpy} -m ipykernel install --user --name {ENV} --display-name "Python ({ENV})"')
    print("  (install ipykernel into the env first if needed)")
    print("Then Kernel -> Change Kernel -> Python (cdv-phylo)")

In [ ]:
from pathlib import Path

CFG = ROOT / "config/lineage_references.tsv"
ACC = "AF259552"

lines = CFG.read_text().splitlines()
hit = [l for l in lines if l.startswith(ACC)]
print("removing:", hit or "NOT FOUND — check the accession")

kept = []
for l in lines:
    if l.startswith(ACC):
        kept.append(
            f"# REMOVED {ACC}\tAmerica-2 (as listed)\t(reference)\tnuc\t"
            f"Sourced from a patent filing. Excluded after the reference tree placed it "
            f"at patristic distance 0.106 from the query set — as far as the vaccine "
            f"strains — indicating the lineage label is wrong."
        )
    else:
        kept.append(l)

CFG.write_text("\n".join(kept) + "\n")

active = [l for l in CFG.read_text().splitlines()
          if l.strip() and not l.strip().startswith("#")]
print(f"\n{len(active)} active references remaining")
from collections import Counter
print(dict(Counter(l.split('\t')[1] for l in active if '\t' in l)))

In [ ]:
import os, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "scripts").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)

RUNDIR = ROOT / "beast/clade3/run100M_fixed"
TOTAL  = 100_000_000
SPAN   = 31.6          # sampling span, for the sanity check

drv = RUNDIR / "driver.log"
print(drv.read_text().strip() if drv.exists() else "no driver log yet")

for sd in sorted(RUNDIR.glob("seed*")):
    print(f"\n{'='*54}\n{sd.name}\n{'='*54}")
    log, out = sd / "clade3_5state.log", sd / "run.out"
    if not out.exists():
        print("  not started"); continue
    if not log.exists():
        print("  starting up, no log yet"); continue

    rows = [l for l in log.read_text().splitlines() if l and not l.startswith("#")]
    hdr  = rows[0].split("\t")
    data = [r.split("\t") for r in rows[1:] if r.split("\t")[0].isdigit()]
    if not data:
        print("  no samples yet"); continue

    state = int(data[-1][0])
    done  = state >= TOTAL * 0.999
    age   = (time.time() - out.stat().st_mtime) / 60

    print(f"  state {state:,} / {TOTAL:,}  ({state/TOTAL:.1%})   {len(data)} samples")
    print(f"  {'COMPLETE' if done else 'running'} — last write {age:.0f} min ago")
    if not done:
        elapsed = (time.time() - (out.stat().st_ctime)) / 60
        if state > 0 and elapsed > 1:
            rate = state / elapsed
            print(f"  rate ~{rate/1e6:.2f} M/min, ETA {(TOTAL-state)/rate/60:.1f} h")
    if not done and age > 25:
        print("  !! no output for 25+ min — check it hasn't stalled")

    # sanity: tree height must be plausible against the sampling span
    if "Tree.height" in hdr:
        i = hdr.index("Tree.height")
        h = [float(r[i]) for r in data]
        recent = h[len(h)//2:]
        print(f"\n  Tree.height  range {min(h):.1f} - {max(h):.1f}, "
              f"recent mean {sum(recent)/len(recent):.1f}")
        mean_h = sum(recent)/len(recent)
        if mean_h < SPAN:
            print("  !! shorter than the sampling span — impossible, check dates")
        elif mean_h > SPAN * 4:
            print("  !! much larger than expected — check the date direction")
        else:
            print("  OK — plausible for a 31.6-year sampling span")

    if "ucldMean.H_clade_3_5state" in hdr:
        i = hdr.index("ucldMean.H_clade_3_5state")
        v = [float(r[i]) for r in data][len(data)//2:]
        print(f"  ucldMean     recent mean {sum(v)/len(v):.3e}  "
              f"(TempEst gave 7.46e-04)")

    for f in sorted(sd.glob("*")):
        if f.suffix in (".log", ".trees"):
            print(f"    {f.name:<38} {f.stat().st_size/1024:>9.1f} KB")

In [ ]:
from pathlib import Path
import time

sd = ROOT / "beast/clade3/run100M_fixed/seed54321"
log = sd / "clade3_5state.log"
if not log.exists():
    print("chain 2 hasn't started or hasn't written yet")
else:
    rows = [l for l in log.read_text().splitlines()
            if l and not l.startswith("#") and l.split()[0].isdigit()]
    state = int(rows[-1].split()[0])
    started = (sd / "run.out").stat().st_ctime
    elapsed = (time.time() - started) / 60
    rate = state / elapsed
    left = (100_000_000 - state) / rate
    print(f"state {state:,} ({state/1e8:.1%}) after {elapsed/60:.1f} h")
    print(f"~{left/60:.1f} h remaining")
    print(f"chain done ~{time.strftime('%a %H:%M', time.localtime(time.time()+left*60))}")
    print(f"MCC tree ~{time.strftime('%a %H:%M', time.localtime(time.time()+(left+15)*60))}")

In [ ]:
import os, re, shutil, subprocess, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "scripts").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)

SD  = ROOT / "beast/clade3/run100M_fixed/seed12345"
MCC = SD / "clade3_mcc_seed12345.tree"

def find_tool(name):
    p = shutil.which(name)
    if p: return p
    for rt in [Path("/Applications"), Path.home()/"Applications", Path.home()]:
        if rt.is_dir():
            for pat in (f"BEAST*/bin/{name}", f"*/BEAST*/bin/{name}"):
                hits = sorted(rt.glob(pat))
                if hits: return str(hits[0])
    raise RuntimeError(f"{name} not found")

ta = find_tool("treeannotator")
trees = SD / "clade3_5state.host.trees"
print(f"input : {trees.name}  ({trees.stat().st_size/1e6:.1f} MB)")

cmd = [ta, "-burnin", "10", "-height", "median", str(trees), str(MCC)]
print("$ " + " ".join(cmd) + "\n")
t0 = time.time()
p = subprocess.run(cmd, cwd=str(SD), capture_output=True, text=True)
print((p.stdout + p.stderr)[-1500:])
print(f"exit {p.returncode} in {(time.time()-t0)/60:.1f} min")
if p.returncode != 0:
    raise RuntimeError("TreeAnnotator failed")

# ---------- verify the time axis before anything else ----------
raw = MCC.read_text()
tr = {}
m = re.search(r"Translate(.*?);", raw, re.S)
for line in m.group(1).replace("\n", " ").split(","):
    q = line.split()
    if len(q) >= 2: tr[q[0].strip()] = q[1].strip().strip("'\"")

nwk = [l for l in raw.splitlines() if l.strip().startswith("tree ")][0]
dates, bad = {}, 0
for num, lab in tr.items():
    mm = re.search(rf"[(,]{num}\[&([^\]]*)\]", nwk)
    if not mm: continue
    h = re.search(r'(?<![\w.])height=([0-9.E+-]+)', mm.group(1))
    if not h: continue
    dates[lab] = (float(lab.split("|")[-1]), float(h.group(1)))

newest = max(d for d, _ in dates.values())
print(f"\nTIME AXIS CHECK  (most recent sample {newest})")
for lab, (date, got) in sorted(dates.items(), key=lambda kv: -kv[1][0])[:3]:
    exp = newest - date
    ok = abs(exp - got) < 0.05
    bad += not ok
    print(f"  {lab.split('|')[0]:<11} date {date:9.3f}  expected h {exp:6.2f}  got {got:6.2f}  {'OK' if ok else 'MISMATCH'}")
for lab, (date, got) in sorted(dates.items(), key=lambda kv: kv[1][0])[:2]:
    exp = newest - date
    ok = abs(exp - got) < 0.05
    bad += not ok
    print(f"  {lab.split('|')[0]:<11} date {date:9.3f}  expected h {exp:6.2f}  got {got:6.2f}  {'OK' if ok else 'MISMATCH'}")

if bad:
    raise RuntimeError(f"{bad} height mismatches — time axis wrong, do not interpret")
print("\nTIME AXIS CORRECT")

root_h = max(float(x) for x in re.findall(r'(?<![\w.])height=([0-9.]+)', raw))
print(f"root height {root_h:.1f} yr  ->  TMRCA {newest - root_h:.0f}")
print(f"\nwrote {MCC}")

In [ ]:
from Bio import Entrez, SeqIO
Entrez.email = "<put your NCBI-registered email here>"

ACCS = ["MW984527","MW984530","MW984531","MW984532","MW984535",
        "MT932504","MT932511","MT932505"]

h = Entrez.efetch(db="nuccore", id=",".join(ACCS), rettype="gb", retmode="text")
for r in SeqIO.parse(h, "genbank"):
    src = next((f for f in r.features if f.type == "source"), None)
    q = src.qualifiers if src else {}
    print("="*66)
    print(f"{r.id}  {r.description[:70]}")
    print(f"  host      : {q.get('host',['-'])[0]}")
    print(f"  country   : {q.get('geo_loc_name', q.get('country',['-']))[0]}")
    print(f"  date      : {q.get('collection_date',['-'])[0]}")
    print(f"  isolate   : {q.get('isolate', q.get('strain',['-']))[0]}")
    for ref in r.annotations.get("references", []):
        if ref.title and ref.title != "Direct Submission":
            print(f"  TITLE     : {ref.title}")
            print(f"  journal   : {ref.journal}")
            print(f"  authors   : {ref.authors[:100]}")
        elif ref.title == "Direct Submission":
            print(f"  submitted : {ref.journal[:110]}")
h.close()

In [ ]:
from pathlib import Path
import time

sd = ROOT / "beast/clade3/run100M_fixed/seed54321"
log = sd / "clade3_5state.log"

def state():
    rows = [l for l in log.read_text().splitlines()
            if l and not l.startswith("#") and l.split()[0].isdigit()]
    return int(rows[-1].split()[0])

s0, t0 = state(), time.time()
print(f"state {s0:,} ({s0/1e8:.1%}) — measuring for 5 minutes...")
time.sleep(300)
s1, t1 = state(), time.time()

rate = (s1 - s0) / ((t1 - t0) / 60)
left = (100_000_000 - s1) / rate
print(f"state {s1:,}")
print(f"rate  {rate/1e6:.3f} M/min  ({1/(rate/1e6):.2f} min per M)")
print(f"~{left/60:.1f} h remaining")
print(f"done around {time.strftime('%a %H:%M', time.localtime(time.time()+left*60))}")

In [ ]:
import os, re, shutil, subprocess, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "scripts").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)

RUN = ROOT / "beast/clade3/run100M_fixed"
SEEDS = [12345, 54321]

def find_tool(name):
    p = shutil.which(name)
    if p: return p
    for rt in [Path("/Applications"), Path.home()/"Applications", Path.home()]:
        if rt.is_dir():
            for pat in (f"BEAST*/bin/{name}", f"*/BEAST*/bin/{name}"):
                hits = sorted(rt.glob(pat))
                if hits: return str(hits[0])
    raise RuntimeError(f"{name} not found")

lc, ta = find_tool("logcombiner"), find_tool("treeannotator")

# confirm both chains finished
for s in SEEDS:
    log = RUN / f"seed{s}" / "clade3_5state.log"
    rows = [l for l in log.read_text().splitlines()
            if l and not l.startswith("#") and l.split()[0].isdigit()]
    st = int(rows[-1].split()[0])
    print(f"seed{s}: {len(rows)} samples, final state {st:,}"
          + ("  COMPLETE" if st >= 99_000_000 else "  INCOMPLETE"))
    assert st >= 99_000_000

def run(cmd, label):
    print(f"\n=== {label} ===\n$ " + " ".join(str(c) for c in cmd))
    t0 = time.time()
    p = subprocess.run([str(c) for c in cmd], cwd=str(RUN), capture_output=True, text=True)
    out = (p.stdout or "") + (p.stderr or "")
    print(out[-800:] if p.returncode == 0 else out[-2500:])
    print(f"exit {p.returncode} in {(time.time()-t0)/60:.1f} min")
    if p.returncode: raise RuntimeError(f"{label} failed")

run([lc, "-log", f"seed{SEEDS[0]}/clade3_5state.log",
         "-log", f"seed{SEEDS[1]}/clade3_5state.log",
         "-b", 10, "-o", "combined_fixed.log"], "LogCombiner: parameters")

run([lc, "-log", f"seed{SEEDS[0]}/clade3_5state.host.trees",
         "-log", f"seed{SEEDS[1]}/clade3_5state.host.trees",
         "-b", 10, "-o", "combined_fixed.host.trees"], "LogCombiner: host trees")

# burnin 0 — LogCombiner already removed it from each chain
run([ta, "-burnin", 0, "-height", "median",
     "combined_fixed.host.trees", "clade3_mcc_fixed.tree"], "TreeAnnotator")

# ---------- verify the time axis before interpreting anything ----------
MCC = RUN / "clade3_mcc_fixed.tree"
raw = MCC.read_text()
tr = {}
m = re.search(r"Translate(.*?);", raw, re.S)
for line in m.group(1).replace("\n", " ").split(","):
    q = line.split()
    if len(q) >= 2: tr[q[0].strip()] = q[1].strip().strip("'\"")
nwk = [l for l in raw.splitlines() if l.strip().startswith("tree ")][0]

dates, bad = {}, 0
for num, lab in tr.items():
    mm = re.search(rf"[(,]{num}\[&([^\]]*)\]", nwk)
    if not mm: continue
    h = re.search(r'(?<![\w.])height=([0-9.E+-]+)', mm.group(1))
    if h: dates[lab] = (float(lab.split("|")[-1]), float(h.group(1)))

newest = max(d for d, _ in dates.values())
print(f"\nTIME AXIS CHECK (most recent {newest})")
for lab, (dte, got) in sorted(dates.items(), key=lambda kv: -kv[1][0])[:2] + \
                      sorted(dates.items(), key=lambda kv: kv[1][0])[:2]:
    exp = newest - dte
    ok = abs(exp - got) < 0.05; bad += not ok
    print(f"  {lab.split('|')[0]:<11} {dte:9.3f}  expected {exp:6.2f}  got {got:6.2f}  {'OK' if ok else 'MISMATCH'}")
if bad: raise RuntimeError("time axis wrong — do not interpret")

root_h = max(float(x) for x in re.findall(r'(?<![\w.])height=([0-9.]+)', raw))
print(f"\nTIME AXIS CORRECT")
print(f"root height {root_h:.1f} yr -> TMRCA {newest - root_h:.0f}")

rows = [l for l in (RUN/"combined_fixed.log").read_text().splitlines()
        if l and not l.startswith("#") and l.split()[0].isdigit()]
print(f"combined trace: {len(rows)} samples (expect ~1800)")
print(f"\nwrote {MCC}")

In [ ]:
import subprocess, shutil
from pathlib import Path

MCC = ROOT / "beast/clade3/run100M_fixed/clade3_mcc_fixed.tree"
NS  = ROOT / "nextstrain"; NS.mkdir(exist_ok=True)

augur = shutil.which("augur")
if not augur:
    for r in [Path.home()/"anaconda3", Path.home()/"miniforge3"]:
        hits = sorted(r.glob("envs/cdv-phylo/bin/augur"))
        if hits: augur = str(hits[0]); break
if not augur:
    raise RuntimeError("augur not found — install it first")

p = subprocess.run([augur, "import", "beast",
                    "--mcc", str(MCC),
                    "--output-tree", str(NS/"tree.nwk"),
                    "--output-node-data", str(NS/"branch_lengths.json"),
                    "--recursion-limit", "10000"],
                   cwd=str(ROOT), capture_output=True, text=True)
print(p.stdout[-1500:] or p.stderr[-2000:])
print("exit", p.returncode)
for f in ["tree.nwk", "branch_lengths.json"]:
    q = NS/f
    print(f"  {f:<24} {q.stat().st_size/1024:>8.1f} KB" if q.exists() else f"  {f}  MISSING")

In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "scripts").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("repo root:", ROOT)

MCC  = ROOT / "beast/clade3/run100M_fixed/clade3_mcc_fixed.tree"
SUB  = ROOT / "data/processed/H_clade_3_metadata.tsv"
FULL = ROOT / "data/processed/metadata_clean.tsv"
OUT  = ROOT / "auspice/cdv_clade3.json"

for f, name in [(MCC, "MCC tree"), (SUB, "subset metadata"), (FULL, "full metadata")]:
    print(f"  {name:<18} {'OK  ' if f.is_file() else 'MISSING'}  {f.relative_to(ROOT)}")
if not MCC.is_file():
    raise FileNotFoundError("MCC tree missing")
if not SUB.is_file():
    print("\n  metadata files in data/processed:")
    for f in sorted((ROOT/"data/processed").glob("*metadata*")):
        print("   ", f.name)
    raise FileNotFoundError("adjust SUB to the right filename")

# use the conda env's python if this kernel is base
py = sys.executable
if "cdv-phylo" not in py:
    for r in [Path.home()/"anaconda3", Path.home()/"miniforge3", Path.home()/"miniconda3"]:
        hits = sorted(r.glob("envs/cdv-phylo/bin/python"))
        if hits:
            py = str(hits[0]); break
print(f"\npython: {py}")

cmd = [py, "scripts/09_make_auspice.py",
       "--mcc", str(MCC.relative_to(ROOT)),
       "--metadata", str(SUB.relative_to(ROOT)),
       "--full-metadata", str(FULL.relative_to(ROOT)),
       "--most-recent", "2023.622",
       "--title", "CDV America-2 lineage in North American carnivores",
       "--maintainer", "Batya Nightingale",
       "--output", str(OUT.relative_to(ROOT))]

print("$ " + " ".join(cmd) + "\n")
p = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
print(p.stdout or "")
if p.returncode:
    print("STDERR:", p.stderr[-2000:])
else:
    import json
    d = json.load(OUT.open())
    def walk(n):
        yield n
        for c in n.get("children", []): yield from walk(c)
    tips = [n for n in walk(d["tree"]) if "children" not in n]
    withc = sum(1 for t in tips if "country" in t["node_attrs"])
    print(f"tips with country: {withc}/{len(tips)}")
    if withc:
        from collections import Counter
        print("countries:", dict(Counter(t["node_attrs"]["country"]["value"] for t in tips
                                         if "country" in t["node_attrs"])))
    print(f"\n{OUT}  ready — drag onto https://auspice.us")

In [ ]:
import os, subprocess, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "scripts").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("repo root:", ROOT)

# use the conda env's python if this kernel is base
py = sys.executable
if "cdv-phylo" not in py:
    for r in [Path.home()/"anaconda3", Path.home()/"miniforge3", Path.home()/"miniconda3"]:
        hits = sorted(r.glob("envs/cdv-phylo/bin/python"))
        if hits:
            py = str(hits[0]); break

JSON = ROOT / "auspice/cdv-phylodynamics.json"
ALN  = ROOT / "data/processed/H_clade_3.fasta"

if not ALN.is_file():
    print("\nFASTA files available:")
    for f in sorted((ROOT/"data/processed").glob("*.fasta")):
        print("  ", f.name)
    raise FileNotFoundError("set ALN to the alignment your tree was built from")

cmd = [py, "scripts/10_add_entropy.py",
       "--json", str(JSON.relative_to(ROOT)),
       "--alignment", str(ALN.relative_to(ROOT)),
       "--gene", "H",
       "--output", str(JSON.relative_to(ROOT))]
print("$ " + " ".join(cmd) + "\n")
p = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
print(p.stdout or "")
if p.returncode:
    print("STDERR:", p.stderr[-2000:])

In [ ]:
# 1. rebuild the clean Auspice JSON
run_script("09_make_auspice.py",
           "--mcc", "beast/clade3/run100M_fixed/clade3_mcc_fixed.tree",
           "--metadata", "data/processed/H_clade_3_metadata.tsv",
           "--full-metadata", "data/processed/metadata_clean.tsv",
           "--most-recent", "2023.622",
           "--title", "CDV America-2 lineage in North American carnivores",
           "--maintainer", "Batya Nightingale",
           "--output", "auspice/cdv-phylodynamics.json")

# 2. then add entropy
run_script("10_add_entropy.py",
           "--json", "auspice/cdv-phylodynamics.json",
           "--alignment", "data/processed/H_clade_3.fasta",
           "--gene", "H",
           "--output", "auspice/cdv-phylodynamics.json")

In [ ]:
import os, subprocess, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "scripts").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("repo root:", ROOT)

py = sys.executable
if "cdv-phylo" not in py:
    for r in [Path.home()/"anaconda3", Path.home()/"miniforge3", Path.home()/"miniconda3"]:
        hits = sorted(r.glob("envs/cdv-phylo/bin/python"))
        if hits:
            py = str(hits[0]); break
print("python   :", py, "\n")

def run(script, *args):
    cmd = [py, f"scripts/{script}", *map(str, args)]
    print("$ " + " ".join(cmd))
    p = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
    print(p.stdout or "")
    if p.stderr.strip():
        print("STDERR:", p.stderr.strip()[-1500:])
    print("-"*60)
    return p.returncode

# 1. rebuild the Auspice JSON from scratch (the current one has bad mutation data)
rc = run("09_make_auspice.py",
         "--mcc", "beast/clade3/run100M_fixed/clade3_mcc_fixed.tree",
         "--metadata", "data/processed/H_clade_3_metadata.tsv",
         "--full-metadata", "data/processed/metadata_clean.tsv",
         "--most-recent", "2023.622",
         "--title", "CDV America-2 lineage in North American carnivores",
         "--maintainer", "Batya Nightingale",
         "--output", "auspice/cdv-phylodynamics.json")

# 2. add the entropy panel
if rc == 0:
    run("10_add_entropy.py",
        "--json", "auspice/cdv-phylodynamics.json",
        "--alignment", "data/processed/H_clade_3.fasta",
        "--gene", "H",
        "--output", "auspice/cdv-phylodynamics.json")

In [ ]:
from pathlib import Path
t = (ROOT / "scripts/10_add_entropy.py").read_text()
print("has BASES set      :", "BASES = set" in t)
print("has plausibility chk:", "per branch" in t)
print("gap handling fixed  :", "c if c not in AMBIG else None" in t)

In [ ]:
rc = run("09_make_auspice.py",
         "--mcc", "beast/clade3/run100M_fixed/clade3_mcc_fixed.tree",
         "--metadata", "data/processed/H_clade_3_metadata.tsv",
         "--full-metadata", "data/processed/metadata_clean.tsv",
         "--most-recent", "2023.622",
         "--title", "CDV America-2 lineage in North American carnivores",
         "--maintainer", "Batya Nightingale",
         "--output", "auspice/cdv-phylodynamics.json")

if rc == 0:
    run("10_add_entropy.py",
        "--json", "auspice/cdv-phylodynamics.json",
        "--alignment", "data/processed/H_clade_3.fasta",
        "--gene", "H",
        "--output", "auspice/cdv-phylodynamics.json")

In [ ]:
run("11_annotate_H.py",
    "--json", "auspice/cdv-phylodynamics.json",
    "--alignment", "data/processed/H_clade_3.fasta",
    "--auto-reference",
    "--output", "auspice/cdv-phylodynamics.json")

In [ ]:
run("11_annotate_H.py",
    "--json", "auspice/cdv-phylodynamics.json",
    "--alignment", "data/processed/H_clade_3.fasta",
    "--auto-reference",
    "--output", "auspice/cdv-phylodynamics.json")

In [ ]:
run("08_add_references.py",
    "--email", "<put your NCBI-registered email here>",
    "--query-fasta", "data/processed/H_clade_3.fasta",
    "--prefix", "H_clade3_with_A75")

In [ ]:
from pathlib import Path
for f in sorted((ROOT/"data/processed").glob("*.fasta")):
    seqs = [l for l in f.read_text().splitlines() if not l.startswith(">")]
    print(f"  {f.name:<40} {len(''.join(seqs))//max(1,sum(1 for l in f.read_text().splitlines() if l.startswith('>')))} nt/seq")

In [ ]:

run("11_annotate_H.py",
    "--json", "auspice/cdv-phylodynamics.json",
    "--alignment", ALN,
    "--reference", "REF_America-2_A75-17_AF164967|reference|0000",
    "--output", "auspice/cdv-phylodynamics.json")

In [ ]:
import json
from pathlib import Path

J = ROOT / "auspice/cdv-phylodynamics.json"
d = json.loads(J.read_text())
ann = d["meta"]["genome_annotations"]

# span from the first to the last domain feature = the H CDS
feats = [(v["start"], v["end"]) for k, v in ann.items() if k != "nuc"]
start, end = min(s for s, _ in feats), max(e for _, e in feats)

d["meta"]["genome_annotations"] = {
    "nuc": ann["nuc"],
    "H": {"start": start, "end": end, "strand": "+", "type": "CDS"},
}
J.write_text(json.dumps(d, indent=1))
print(f"replaced {len(feats)} domain features with a single H gene: {start}-{end}")

In [ ]:
import json
from pathlib import Path

J = ROOT / "auspice/cdv-phylodynamics.json"
print("file:", J, f"{J.stat().st_size/1024:.0f} KB")
d = json.loads(J.read_text())

print("panels:", d["meta"].get("panels"))
ann = d["meta"].get("genome_annotations")
print("\ngenome_annotations present:", ann is not None)
if ann:
    print(json.dumps(ann, indent=1))

# does the tree still carry mutations?
def walk(n):
    yield n
    for c in n.get("children", []): yield from walk(c)
nodes = list(walk(d["tree"]))
withm = sum(1 for n in nodes
            if n.get("branch_attrs", {}).get("mutations", {}).get("nuc"))
print(f"\nnodes with nuc mutations: {withm}/{len(nodes)}")

In [ ]:
ALN = "data/processed/clade3_with_refs_dedup.fasta"

run("09_make_auspice.py",
    "--mcc", "beast/clade3/run100M_fixed/clade3_mcc_fixed.tree",
    "--metadata", "data/processed/H_clade_3_metadata.tsv",
    "--full-metadata", "data/processed/metadata_clean.tsv",
    "--most-recent", "2023.622",
    "--title", "CDV America-2 lineage in North American carnivores",
    "--maintainer", "Batya Nightingale",
    "--output", "auspice/cdv-phylodynamics.json")

run("10_add_entropy.py",
    "--json", "auspice/cdv-phylodynamics.json",
    "--alignment", ALN,
    "--gene", "H",
    "--cds", "21", "1836",
    "--output", "auspice/cdv-phylodynamics.json")

run("11_annotate_H.py",
    "--json", "auspice/cdv-phylodynamics.json",
    "--alignment", ALN,
    "--reference", "REF_America-2_A75-17_AF164967|reference|0000",
    "--output", "auspice/cdv-phylodynamics.json")

In [ ]:
run("11_annotate_H.py",
    "--json", "auspice/cdv-phylodynamics.json",
    "--alignment", "data/processed/clade3_with_refs_dedup.fasta",
    "--reference", "REF_America-2_A75-17_AF164967|reference|0000",
    "--output", "auspice/cdv-phylodynamics.json")

In [ ]:
ALN = "data/processed/clade3_with_refs_dedup.fasta"

run("09_make_auspice.py",
    "--mcc", "beast/clade3/run100M_fixed/clade3_mcc_fixed.tree",
    "--metadata", "data/processed/H_clade_3_metadata.tsv",
    "--full-metadata", "data/processed/metadata_clean.tsv",
    "--most-recent", "2023.622",
    "--title", "CDV America-2 lineage in North American carnivores",
    "--maintainer", "Batya Nightingale",
    "--output", "auspice/cdv-phylodynamics.json")

run("11_annotate_H.py",
    "--json", "auspice/cdv-phylodynamics.json",
    "--alignment", ALN,
    "--reference", "REF_America-2_A75-17_AF164967|reference|0000",
    "--domains",
    "--output", "auspice/cdv-phylodynamics.json")

run("10_add_entropy.py",
    "--json", "auspice/cdv-phylodynamics.json",
    "--alignment", ALN,
    "--cds-from-json",
    "--output", "auspice/cdv-phylodynamics.json")

In [ ]:
run("11_annotate_H.py",
    "--json", "auspice/cdv-phylodynamics.json",
    "--alignment", "data/processed/clade3_with_refs_dedup.fasta",
    "--reference", "REF_America-2_A75-17_AF164967|reference|0000",
    "--domains",
    "--output", "auspice/cdv-phylodynamics.json")

In [ ]:
run("11_annotate_H.py",
    "--json", "auspice/cdv-phylodynamics.json",
    "--alignment", "data/processed/clade3_with_refs_dedup.fasta",
    "--reference", "REF_America-2_A75-17_AF164967|reference|0000",
    "--domains",
    "--output", "auspice/cdv-phylodynamics.json")

run("10_add_entropy.py",
    "--json", "auspice/cdv-phylodynamics.json",
    "--alignment", "data/processed/clade3_with_refs_dedup.fasta",
    "--cds-from-json",
    "--output", "auspice/cdv-phylodynamics.json")

In [ ]:
run("11_annotate_H.py",
    "--json", "auspice/cdv-phylodynamics.json",
    "--alignment", "data/processed/clade3_with_refs_dedup.fasta",
    "--reference", "REF_America-2_A75-17_AF164967|reference|0000",
    "--domains",
    "--max-feature-aa", "60",
    "--output", "auspice/cdv-phylodynamics.json")

run("10_add_entropy.py",
    "--json", "auspice/cdv-phylodynamics.json",
    "--alignment", "data/processed/clade3_with_refs_dedup.fasta",
    "--cds-from-json",
    "--output", "auspice/cdv-phylodynamics.json")

In [ ]:
run("11_annotate_H.py",
    "--json", "auspice/cdv-phylodynamics.json",
    "--alignment", "data/processed/clade3_with_refs_dedup.fasta",
    "--reference", "REF_America-2_A75-17_AF164967|reference|0000",
    "--domains",
    "--output", "auspice/cdv-phylodynamics.json")

run("10_add_entropy.py",
    "--json", "auspice/cdv-phylodynamics.json",
    "--alignment", "data/processed/clade3_with_refs_dedup.fasta",
    "--cds-from-json",
    "--output", "auspice/cdv-phylodynamics.json")

In [ ]:
run("11_annotate_H.py",
    "--json", "auspice/cdv-phylodynamics.json",
    "--alignment", "data/processed/clade3_with_refs_dedup.fasta",
    "--reference", "REF_America-2_A75-17_AF164967|reference|0000",
    "--domains",
    "--max-feature-aa", "0",
    "--output", "auspice/cdv-phylodynamics.json")

run("10_add_entropy.py",
    "--json", "auspice/cdv-phylodynamics.json",
    "--alignment", "data/processed/clade3_with_refs_dedup.fasta",
    "--cds-from-json",
    "--output", "auspice/cdv-phylodynamics.json")